In [1]:
import heapq
from bisect import bisect_right
from collections import OrderedDict, defaultdict, deque

# 1) Stack

**In plain English.** A stack remembers things in the order you will need to come back to
them: the most recent unfinished item is always on top.

> **Analogy.** A pile of plates: you can only take the top one. Opening brackets are plates
> you put down; a closing bracket must take the top plate, and it had better be its partner.

Two families:

| Family | Stack holds | Pops when |
|---|---|---|
| **Matching / evaluation** | open brackets, partial results, saved context | the matching closer or operator arrives |
| **Monotonic stack** | indices whose values are increasing (or decreasing) | a new value breaks the order — every popped element has just found its *next greater / smaller* element |

```python
stack = []                          # indices, values decreasing
for i, x in enumerate(nums):
    while stack and nums[stack[-1]] < x:
        j = stack.pop()             # x is the next greater element of nums[j]
    stack.append(i)
```

**Recognition signals**

- brackets, "valid", "balanced", nested `k[...]`, paths with `..`
- evaluate an expression, "calculator", postfix
- "next greater / warmer / smaller", "previous greater", "span", "how many days until"
- histogram, "largest rectangle", "sum of subarray minimums"
- "remove k digits / letters to get the smallest result" (greedy monotonic stack)

**Common bugs**

- Popping from an empty stack — always check `stack` before `stack[-1]` or `pop()`.
- Forgetting the leftovers after the loop (unmatched `(`, the last number, a flush sentinel).
- Storing values when you need *distances* — push indices instead.
- `>=` vs `>` in monotonic stacks: decides how equal values are handled; with contribution
  counting, one side must be strict and the other not.
- `//` floors negatives; truncation toward zero needs `int(a / b)`.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Queue / deque | FIFO; the oldest item leaves first (BFS, sliding window max) |
| Monotonic deque | a monotonic stack that *also* drops items from the front when they leave a window |
| Recursion | uses the call stack implicitly; an explicit stack is the iterative version |
| Two pointers | moves over the data once; a stack lets you *return* to earlier unfinished items |

**Learning path.** Matching (Valid Parentheses) → stacks with extra info (Min Stack) →
evaluation (RPN, Calculator II, Calculator) → saving context on `[` (Decode String) →
collisions and cancellations → parentheses repair and longest valid run → a stack per
frequency (FreqStack). Then the monotonic stack: next greater (Daily Temperatures and
variants) → spans → fleets → histogram rectangle → greedy smallest-result → contribution
counting.

## 1.1 Matching and evaluation

Ask: *what is the most recent unfinished thing, and what finishes it?* The answer tells you
what to push and when to pop. When nesting appears (`(`, `[`), push the whole current
context and start fresh.

### 1.1.1 [20. Valid Parentheses](https://leetcode.com/problems/valid-parentheses/) — ★★★ · Easy

**Problem.** Is a string of `()[]{}` correctly matched and nested?  
**Signal.** Most recent open bracket must close first → stack.  
*New here: The basic push/pop matching loop.*

**Fine print.** Ask before coding:
- Only the six bracket characters? → Yes; other characters would need to be skipped.
- Is an empty string valid? → Yes.

> 💡 **Key insight.** The bracket that must close next is always the most recently opened one — exactly the top of a stack.

**Brute force →** Repeatedly delete `"()"`, `"[]"`, `"{}"` until nothing changes → O(n²). A stack checks each bracket once.

**Steps**
1. Push every opening bracket.
2. On a closing bracket, the stack must be non-empty and its top must be the partner.
3. Pop the partner and continue.
4. Valid iff the stack is empty at the end.

**Complexity**

- **Time:** O(n) — each character pushed/popped at most once.
- **Space:** O(n) — all openers in the worst case.

In [2]:
PAIRS = {")": "(", "]": "[", "}": "{"}


def is_valid(s: str) -> bool:
    """True if brackets are balanced and properly nested."""
    stack = []
    for ch in s:
        if ch in PAIRS:
            # closer must match the most recent unclosed opener
            if not stack or stack.pop() != PAIRS[ch]:
                return False
        else:
            stack.append(ch)
    # leftover openers were never closed
    return not stack


assert is_valid("()[]{}")
assert not is_valid("(]")
assert is_valid("([])")
assert not is_valid("([)]")
assert not is_valid("(")
print(is_valid("{[()]}"))

True


### 1.1.2 [155. Min Stack](https://leetcode.com/problems/min-stack/) — ★★ · Medium

**Problem.** Stack with `push`, `pop`, `top` and `get_min`, all O(1).  
**Signal.** Minimum must survive pops → store the running minimum alongside each element.  
*New here: Storing extra information *with* each stack entry.*

**Fine print.** Ask before coding:
- Will `pop`, `top` or `get_min` be called on an empty stack? → No, calls are always valid.
- Duplicate minimums? → Yes, so popping one copy must not lose the minimum.

> 💡 **Key insight.** The minimum of everything below an entry cannot change while that entry is on the stack, so it can be stored with the entry once.

**Brute force →** Scan the stack for the minimum on every `get_min` → O(n). Storing `(value, min_so_far)` makes it O(1).

**Steps**
1. Each entry is `(value, minimum of the stack up to here)`.
2. `push`: the new minimum is `min(val, previous minimum)`.
3. `pop` removes the pair, which automatically restores the older minimum.
4. `get_min` reads the top pair.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(n) — one extra number per entry.

In [3]:
class MinStack:
    """Stack that reports its minimum in O(1)."""

    def __init__(self) -> None:
        # (value, minimum of the stack up to and including this entry)
        self.items: list[tuple[int, int]] = []

    def push(self, val: int) -> None:
        low = min(val, self.items[-1][1]) if self.items else val
        self.items.append((val, low))

    def pop(self) -> None:
        # the entry below already stores the older minimum
        self.items.pop()

    def top(self) -> int:
        return self.items[-1][0]

    def get_min(self) -> int:
        return self.items[-1][1]


st = MinStack()
for v in [-2, 0, -3]:
    st.push(v)
assert st.get_min() == -3
st.pop()
assert st.top() == 0 and st.get_min() == -2
print(st.get_min())

-2


### 1.1.3 [150. Evaluate Reverse Polish Notation](https://leetcode.com/problems/evaluate-reverse-polish-notation/) — ★★ · Medium

**Problem.** Evaluate a postfix expression of integers and `+ - * /` (division truncates toward 0).  
**Signal.** Postfix → operands on a stack, operator pops two.  
*New here: The stack as an evaluator of operands.*

**Fine print.** Ask before coding:
- Is the expression always valid? → Yes, with no division by zero.
- How does division round? → Truncate toward zero, so use `int(a / b)`, not `//`.

> 💡 **Key insight.** In postfix, an operator always applies to the two most recent unconsumed values — the top two of the stack.

**Brute force →** Repeatedly find the first operator and replace it with its result → O(n²). The stack does each token once.

**Steps**
1. Push numbers.
2. On an operator, pop `b` (right operand), then `a` (left operand).
3. Push `a op b`, using `int(a / b)` for division.
4. The final stack value is the answer.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

⚠️ **Trap.** Pop order matters for `−` and `/`; and `//` floors `-7 // 2` to −4 instead of −3.

In [4]:
def eval_rpn(tokens: list[str]) -> int:
    """Value of a postfix expression."""
    stack: list[int] = []
    for tok in tokens:
        if tok in "+-*/":
            # right operand is on top
            b, a = stack.pop(), stack.pop()
            if tok == "+":
                stack.append(a + b)
            elif tok == "-":
                stack.append(a - b)
            elif tok == "*":
                stack.append(a * b)
            else:
                # truncate toward zero, unlike //
                stack.append(int(a / b))
        else:
            stack.append(int(tok))
    return stack[0]


assert eval_rpn(["2", "1", "+", "3", "*"]) == 9
assert eval_rpn(["4", "13", "5", "/", "+"]) == 6
tokens = ["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]
assert eval_rpn(tokens) == 22
print(eval_rpn(tokens))

22


### 1.1.4 [232. Implement Queue using Stacks](https://leetcode.com/problems/implement-queue-using-stacks/) — ★★ · Easy

**Problem.** FIFO queue using only two stacks; amortised O(1) operations.  
**Signal.** Reversing a stack once gives queue order → inbox/outbox.  
*New here: Amortised analysis: an expensive step that happens rarely.*

**Fine print.** Ask before coding:
- Will `pop` or `peek` be called on an empty queue? → No.
- Must every operation be O(1), or amortised O(1)? → Amortised is the target.

> 💡 **Key insight.** Pouring one stack into another reverses it, turning LIFO into FIFO; each element is poured at most once.

**Brute force →** Pour everything back and forth on every pop → O(n) per op. Pouring only when `outbox` is empty → amortised O(1).

**Steps**
1. `push` goes onto `inbox`.
2. `pop`/`peek`: if `outbox` is empty, move all of `inbox` into it.
3. Serve from the top of `outbox`.

**Complexity**

- **Time:** Amortised O(1) — every element is pushed and popped at most twice in total.
- **Space:** O(n).

In [5]:
class MyQueue:
    """Queue built from two stacks."""

    def __init__(self) -> None:
        self.inbox: list[int] = []
        self.outbox: list[int] = []

    def push(self, x: int) -> None:
        self.inbox.append(x)

    def _shift(self) -> None:
        # refill only when empty, reversing inbox into queue order
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())

    def pop(self) -> int:
        self._shift()
        return self.outbox.pop()

    def peek(self) -> int:
        self._shift()
        return self.outbox[-1]

    def empty(self) -> bool:
        return not self.inbox and not self.outbox


q = MyQueue()
q.push(1)
q.push(2)
assert q.peek() == 1 and q.pop() == 1 and not q.empty()
print(q.pop(), q.empty())

2 True


### 1.1.5 [225. Implement Stack using Queues](https://leetcode.com/problems/implement-stack-using-queues/) — ★ · Easy

**Problem.** LIFO stack using only queue operations.  
**Signal.** Rotate the queue after each push so the newest element is at the front.  
*New here: The reverse trick — paying at push time instead.*

**Fine print.** Ask before coding:
- One queue or two? → Either; one queue that rotates after each push is enough.
- Which operations may be expensive? → Make `push` O(n) and the rest O(1), and say why you chose that.

> 💡 **Key insight.** Rotating the queue by `n − 1` after each push puts the newest element at the front, where a queue can remove it.

**Brute force →** Two queues, moving all but the last on every pop → O(n) pop. Rotating on push gives O(1) pop; one side must pay O(n).

**Steps**
1. Append `x` to the queue.
2. Move the `n − 1` older elements from front to back.
3. `pop`/`top` use the front.

**Complexity**

- **Time:** O(n) push, O(1) pop and top.
- **Space:** O(n).

In [6]:
class MyStack:
    """Stack built from one queue."""

    def __init__(self) -> None:
        self.q: deque[int] = deque()

    def push(self, x: int) -> None:
        self.q.append(x)
        # rotate older elements behind x so x is at the front
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())

    def pop(self) -> int:
        return self.q.popleft()

    def top(self) -> int:
        return self.q[0]

    def empty(self) -> bool:
        return not self.q


s = MyStack()
s.push(1)
s.push(2)
assert s.top() == 2 and s.pop() == 2 and not s.empty()
print(s.pop(), s.empty())

1 True


### 1.1.6 [71. Simplify Path](https://leetcode.com/problems/simplify-path/) — ★★ · Medium

**Problem.** Canonicalise a Unix path (handle `.`, `..`, repeated slashes).  
**Signal.** Directory nesting → stack of path components.  
*New here: The stack as a "current location" that can go up a level.*

**Fine print.** Ask before coding:
- Is the path always absolute? → Yes, it starts with `/`.
- Is `...` a directory name? → Yes, only `.` and `..` are special.
- `..` at the root? → Stay at the root.

> 💡 **Key insight.** The stack *is* the current directory path: entering a folder pushes, `..` pops.

**Brute force →** String replacements for `/./` and `/x/../` until stable → fragile and O(n²). Splitting + stack is one pass.

**Steps**
1. Split on `/`.
2. Skip empty parts and `.`.
3. `..` pops if possible; any other name is pushed.
4. Return `"/" + "/".join(stack)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [7]:
def simplify_path(path: str) -> str:
    """Canonical absolute path."""
    stack = []
    for part in path.split("/"):
        # go up one level (ignored at the root)
        if part == "..":
            if stack:
                stack.pop()
        # a real directory name ("" and "." change nothing)
        elif part and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)


assert simplify_path("/home/") == "/home"
assert simplify_path("/home//foo/") == "/home/foo"
assert simplify_path("/home/user/Documents/../Pictures") == "/home/user/Pictures"
assert simplify_path("/../") == "/"
assert simplify_path("/.../a/../b/c/../d/./") == "/.../b/d"
print(simplify_path("/a/./b/../../c/"))

/c


### 1.1.7 [227. Basic Calculator II](https://leetcode.com/problems/basic-calculator-ii/) — ★★ · Medium

**Problem.** Evaluate `+ - * /` with non-negative integers and spaces, no parentheses.  
**Signal.** Precedence without parentheses → apply `*`/`/` immediately, defer `+`/`−` on a stack.  
*New here: Operator precedence: resolve high-precedence operators immediately.*

**Fine print.** Ask before coding:
- Is the expression always valid with no parentheses? → Yes.
- Integer division truncating toward zero? → Yes.
- Are there spaces? → Yes, anywhere; skip them.

> 💡 **Key insight.** Every expression is a sum of terms; `*` and `/` only modify the latest term, so keep terms on a stack and sum them at the end.

**Brute force →** Two passes (first `*`/`/`, then `+`/`−`) on a token list → works, more code. One pass with "the operator before this number" is shorter.

**Steps**
1. Accumulate a number's digits.
2. When an operator or the end arrives, apply the *previous* operator to that number.
3. `+` pushes `num`, `−` pushes `−num`, `*`/`/` replace the top with `top op num`.
4. Return the sum of the stack.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) — O(1) if you keep only the last term and a running sum.

In [8]:
def calculate_ii(s: str) -> int:
    """Evaluate an expression with + - * /."""
    # stack of signed terms; their sum is the answer
    stack: list[int] = []
    num, op = 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        # number finished: apply the operator that came before it
        if ch in "+-*/" or i == len(s) - 1:
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            num, op = 0, ch
    return sum(stack)


assert calculate_ii("3+2*2") == 7
assert calculate_ii(" 3/2 ") == 1
assert calculate_ii(" 3+5 / 2 ") == 5
assert calculate_ii("14-3/2") == 13
print(calculate_ii("3+2*2"))

7


### 1.1.8 [224. Basic Calculator](https://leetcode.com/problems/basic-calculator/) — ★★ · Hard

**Problem.** Evaluate `+`, `−`, parentheses and unary minus.  
**Signal.** Only `+`/`−` → running sum with a sign; parentheses → push context.  
*New here: Saving and restoring *context* (partial result + sign) at each `(`.*

**Fine print.** Ask before coding:
- Is unary minus allowed? → Yes (`-(2+3)`), but unary plus is not.
- Is there multiplication or division? → No, only `+`, `-` and parentheses.
- Can I use `eval`? → No.

> 💡 **Key insight.** A parenthesis is a fresh sub-calculation whose result is added to the outer result with the sign that preceded it, so push `(result, sign)` and start over.

**Brute force →** Recursively evaluate the innermost parentheses and splice the result back → O(n²). One stack pass is O(n).

**Steps**
1. Keep `result`, current `num`, and `sign` (+1/−1).
2. On `+`/`−`, add `sign * num` to `result` and set the new sign.
3. On `(`, push `result` and `sign`, then reset both.
4. On `)`, finish the inner result and combine: `saved_result + saved_sign * inner`.

**Complexity**

- **Time:** O(n).
- **Space:** O(depth) — two numbers per open parenthesis.

In [9]:
def calculate(s: str) -> int:
    """Evaluate an expression with + - and parentheses."""
    stack: list[int] = []
    result, num, sign = 0, 0, 1
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        # close the current number and remember the next sign
        elif ch in "+-":
            result += sign * num
            num, sign = 0, 1 if ch == "+" else -1
        # save the outer context and start a fresh sub-expression
        elif ch == "(":
            stack.extend([result, sign])
            result, sign = 0, 1
        # finish the sub-expression and merge it into the outer one
        elif ch == ")":
            result += sign * num
            num = 0
            saved_sign, saved_result = stack.pop(), stack.pop()
            result = saved_result + saved_sign * result
    return result + sign * num


assert calculate("1 + 1") == 2
assert calculate(" 2-1 + 2 ") == 3
assert calculate("(1+(4+5+2)-3)+(6+8)") == 23
assert calculate("-(2+3)") == -5
print(calculate("(1+(4+5+2)-3)+(6+8)"))

23


### 1.1.9 [394. Decode String](https://leetcode.com/problems/decode-string/) — ★★ · Medium

**Problem.** Decode `k[encoded]` (nested), e.g. `3[a2[c]]` → `accaccacc`.  
**Signal.** Nested brackets → stack of (previous string, repeat count).  
*New here: Context saving where the context is a partial string plus a multiplier.*

**Fine print.** Ask before coding:
- Is the input always valid? → Yes.
- Can `k` have several digits? → Yes, e.g. `12[a]`.
- Do digits appear only as repeat counts? → Yes, never inside the plain text.

> 💡 **Key insight.** At `[` the string built so far and the pending count are paused; at `]` the inner string is repeated and appended to the paused one.

**Brute force →** Repeatedly expand the innermost `k[...]` with string search → O(n · output). The stack builds the output once.

**Steps**
1. Digits accumulate into `count`.
2. `[` pushes `(current, count)` and resets both.
3. `]` pops `(prev, k)` and sets `current = prev + current * k`.
4. Letters append to `current`.

**Complexity**

- **Time:** O(output length).
- **Space:** O(output length).

In [10]:
def decode_string(s: str) -> str:
    """Expand k[...] encodings."""
    stack: list[tuple[str, int]] = []
    current, count = "", 0
    for ch in s:
        # multi-digit repeat counts
        if ch.isdigit():
            count = count * 10 + int(ch)
        # pause the outer string and its pending count
        elif ch == "[":
            stack.append((current, count))
            current, count = "", 0
        # repeat the inner string and attach it to the outer one
        elif ch == "]":
            prev, k = stack.pop()
            current = prev + current * k
        else:
            current += ch
    return current


assert decode_string("3[a]2[bc]") == "aaabcbc"
assert decode_string("3[a2[c]]") == "accaccacc"
assert decode_string("2[abc]3[cd]ef") == "abcabccdcdcdef"
assert decode_string("10[a]") == "a" * 10
print(decode_string("3[a2[c]]"))

accaccacc


### 1.1.10 [735. Asteroid Collision](https://leetcode.com/problems/asteroid-collision/) — ★★ · Medium

**Problem.** Asteroids move right (+) or left (−); colliding ones: smaller explodes, equal both explode. Final state?  
**Signal.** A left-mover only meets the most recent right-movers → stack.  
*New here: A pop loop where the incoming item may also be destroyed.*

**Fine print.** Ask before coding:
- Can a value be 0? → No.
- Do two asteroids moving apart ever collide? → No, only right-moving followed by left-moving.

> 💡 **Key insight.** A left-moving asteroid can only hit right-movers that are still alive to its left, nearest first — the top of the stack.

**Brute force →** Simulate rounds of collisions until stable → O(n²). The stack resolves each asteroid once.

**Steps**
1. For each asteroid, if it moves left and the top moves right, they collide.
2. Top smaller → pop it and keep colliding.
3. Equal → pop and destroy the incoming one too.
4. Top larger → incoming is destroyed. If it survives everything, push it.

**Complexity**

- **Time:** O(n) — each asteroid pushed and popped at most once.
- **Space:** O(n).

In [11]:
def asteroid_collision(asteroids: list[int]) -> list[int]:
    """Surviving asteroids after all collisions."""
    stack: list[int] = []
    for a in asteroids:
        alive = True
        # only a left-mover meeting a right-mover collides
        while alive and a < 0 and stack and stack[-1] > 0:
            # top is smaller: it explodes, keep going
            if stack[-1] < -a:
                stack.pop()
                continue
            # equal: both explode
            if stack[-1] == -a:
                stack.pop()
            alive = False
        if alive:
            stack.append(a)
    return stack


assert asteroid_collision([5, 10, -5]) == [5, 10]
assert asteroid_collision([8, -8]) == []
assert asteroid_collision([10, 2, -5]) == [10]
assert asteroid_collision([-2, -1, 1, 2]) == [-2, -1, 1, 2]
print(asteroid_collision([5, 10, -5]))

[5, 10]


### 1.1.11 [1249. Minimum Remove to Make Valid Parentheses](https://leetcode.com/problems/minimum-remove-to-make-valid-parentheses/) — ★★ · Medium

**Problem.** Remove the fewest parentheses so the string is valid; return any valid result.  
**Signal.** Unmatched `)` are found left-to-right; unmatched `(` are what remain on the stack.  
*New here: Stack of *indices*, so you know which characters to delete.*

**Fine print.** Ask before coding:
- Can there be letters as well as parentheses? → Yes, keep them all.
- Any valid result, or a specific one? → Any.

> 💡 **Key insight.** Every unmatched bracket must go and every matched one can stay: unmatched `)` appear when the stack is empty; unmatched `(` are what is left at the end.

**Brute force →** Try deleting subsets (BFS over deletions) → exponential. One stack pass finds exactly the unmatched ones.

**Steps**
1. Push the index of each `(`.
2. A `)` pops a partner, or is marked for removal if none exists.
3. After the scan, mark every index still on the stack.
4. Build the string without the marked indices.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [12]:
def min_remove_to_make_valid(s: str) -> str:
    """s with the fewest parentheses deleted to balance it."""
    remove, stack = set(), []
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            # matched closer, or one with no partner
            if stack:
                stack.pop()
            else:
                remove.add(i)
    # openers never closed
    remove.update(stack)
    return "".join(ch for i, ch in enumerate(s) if i not in remove)


assert min_remove_to_make_valid("lee(t(c)o)de)") == "lee(t(c)o)de"
assert min_remove_to_make_valid("a)b(c)d") == "ab(c)d"
assert min_remove_to_make_valid("))((") == ""
print(min_remove_to_make_valid("lee(t(c)o)de)"))

lee(t(c)o)de


### 1.1.12 [921. Minimum Add to Make Parentheses Valid](https://leetcode.com/problems/minimum-add-to-make-parentheses-valid/) — ★ · Medium

**Problem.** Fewest insertions to balance a string of `(` and `)`.  
**Signal.** Only counts matter → a balance counter instead of a stack.  
*New here: Replacing a stack by a counter when all items are identical.*

**Fine print.** Ask before coding:
- Only `(` and `)`? → Yes.
- Can you add anywhere? → Yes, at any position.

> 💡 **Key insight.** With one bracket type, the stack only ever holds `(`, so its size (a counter) is all you need.

**Brute force →** Simulate with a stack → O(n) space. A counter → O(1).

**Steps**
1. `open_count` = unmatched `(` so far.
2. `(` → increment; `)` → decrement if possible, else count one insertion.
3. Answer = insertions + remaining `open_count`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [13]:
def min_add_to_make_valid(s: str) -> int:
    """Number of insertions needed to balance s."""
    open_count = inserts = 0
    for ch in s:
        if ch == "(":
            open_count += 1
        # closer matches an earlier opener
        elif open_count:
            open_count -= 1
        # closer with no partner needs an inserted "("
        else:
            inserts += 1
    # each unmatched "(" needs an inserted ")"
    return inserts + open_count


assert min_add_to_make_valid("())") == 1
assert min_add_to_make_valid("(((") == 3
assert min_add_to_make_valid("()))((") == 4
print(min_add_to_make_valid("())"))

1


### 1.1.13 [32. Longest Valid Parentheses](https://leetcode.com/problems/longest-valid-parentheses/) — ★★ · Hard

**Problem.** Length of the longest well-formed parentheses substring.  
**Signal.** Stack of indices with a sentinel marking the last invalid position.  
*New here: A sentinel index on the stack that marks "the last place a valid run could start after".*

**Fine print.** Ask before coding:
- Substring, so contiguous? → Yes.
- Return the length? → Yes.
- Empty string? → Return 0.

> 💡 **Key insight.** After popping a match, the new top is the index just before the current valid run, so the run's length is `i − top`.

**Brute force →** Check every substring with a counter → O(n²). DP is O(n) but subtle; the sentinel stack is O(n) and short.

**Steps**
1. Push `-1` as the base.
2. `(` pushes its index.
3. `)` pops; if the stack is now empty, push `i` as the new base (unmatched `)`).
4. Otherwise update `best = max(best, i − stack[-1])`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [14]:
def longest_valid_parentheses(s: str) -> int:
    """Longest balanced parentheses substring."""
    # bottom of the stack = index just before the current valid run
    stack, best = [-1], 0
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        else:
            stack.pop()
            # unmatched ")": it becomes the new base
            if not stack:
                stack.append(i)
            # valid run from just after the top to i
            else:
                best = max(best, i - stack[-1])
    return best


assert longest_valid_parentheses("(()") == 2
assert longest_valid_parentheses(")()())") == 4
assert longest_valid_parentheses("") == 0
assert longest_valid_parentheses("()(())") == 6
print(longest_valid_parentheses(")()())"))

4


### 1.1.14 [1047. Remove All Adjacent Duplicates In String](https://leetcode.com/problems/remove-all-adjacent-duplicates-in-string/) — ★ · Easy

**Problem.** Repeatedly delete adjacent equal pairs; return the final string.  
**Signal.** Removal can expose a new adjacent pair → stack of kept characters.  
*New here: Cascading cancellations handled by comparing with the top.*

**Fine print.** Ask before coding:
- Only lowercase letters? → Yes.
- Is the final answer unique? → Yes, whatever order the pairs are removed in.

> 💡 **Key insight.** After a pair cancels, the new neighbour of the incoming character is whatever is now on top of the stack.

**Brute force →** Rescan and delete pairs until none remain → O(n²). The stack is one pass.

**Steps**
1. For each character, compare with the top of the stack.
2. Equal → pop (the pair cancels).
3. Different → push.
4. Join the stack.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [15]:
def remove_duplicates_str(s: str) -> str:
    """s after cancelling adjacent equal pairs."""
    stack: list[str] = []
    for ch in s:
        # same as the current neighbour: both vanish
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)


assert remove_duplicates_str("abbaca") == "ca"
assert remove_duplicates_str("azxxzy") == "ay"
print(remove_duplicates_str("abbaca"))

ca


### 1.1.15 [2390. Removing Stars From a String](https://leetcode.com/problems/removing-stars-from-a-string/) — ★ · Medium

**Problem.** Each `*` deletes the closest non-star character to its left; return the result.  
**Signal.** "Closest to the left" = top of a stack.  
*New here: Recognising "closest to the left" as "top of the stack" in disguise.*

**Fine print.** Ask before coding:
- Is there always a non-star character to delete? → Yes, the operation is always valid.

> 💡 **Key insight.** The closest surviving character to the left of a `*` is the top of the stack of kept characters.

**Brute force →** Delete with string slicing each time → O(n²). The stack is O(n).

**Steps**
1. Push letters.
2. A `*` pops.
3. Join the stack.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [16]:
def remove_stars(s: str) -> str:
    """s after applying star deletions."""
    stack: list[str] = []
    for ch in s:
        # a star deletes the nearest kept letter
        if ch == "*":
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)


assert remove_stars("leet**cod*e") == "lecoe"
assert remove_stars("erase*****") == ""
print(remove_stars("leet**cod*e"))

lecoe


### 1.1.16 [895. Maximum Frequency Stack](https://leetcode.com/problems/maximum-frequency-stack/) — ★★ · Hard

**Problem.** `push(x)`; `pop()` removes the most frequent element, ties broken by most recent.  
**Signal.** Group elements by frequency; each frequency level is its own stack.  
*New here: Many stacks, one per frequency level.*

**Fine print.** Ask before coding:
- Will `pop` be called on an empty stack? → No.
- How are frequency ties broken? → By the most recently pushed.

> 💡 **Key insight.** Put an element on stack `f` when it reaches frequency `f`; the top of the highest non-empty stack is the most frequent *and* most recent.

**Brute force →** A heap keyed by `(−freq, −time)` → O(log n) per op. Stacks per frequency give O(1).

**Steps**
1. `freq[x]` counts occurrences; `group[f]` is a stack.
2. `push(x)`: increment `freq[x]` to `f`, push `x` on `group[f]`, update `max_freq`.
3. `pop()`: pop from `group[max_freq]`, decrement its frequency.
4. If that stack is empty, `max_freq −= 1`.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(n) — each push stores one entry.

In [17]:
class FreqStack:
    """Stack popping the most frequent (then most recent) element."""

    def __init__(self) -> None:
        self.freq: dict[int, int] = defaultdict(int)
        # group[f]: elements in the order they reached frequency f
        self.group: dict[int, list[int]] = defaultdict(list)
        self.max_freq = 0

    def push(self, val: int) -> None:
        self.freq[val] += 1
        f = self.freq[val]
        self.max_freq = max(self.max_freq, f)
        self.group[f].append(val)

    def pop(self) -> int:
        # most frequent level, most recent element in it
        val = self.group[self.max_freq].pop()
        self.freq[val] -= 1
        # level emptied: the next level down is now the highest
        if not self.group[self.max_freq]:
            self.max_freq -= 1
        return val


fs = FreqStack()
for v in [5, 7, 5, 7, 4, 5]:
    fs.push(v)
assert [fs.pop() for _ in range(4)] == [5, 7, 5, 4]
print("ok")

ok


## 1.2 Monotonic stack

Keep the stack sorted (increasing or decreasing). When a new element breaks the order,
everything it pops has just met its answer — the next greater (or smaller) element. Each
index is pushed and popped once, so the whole scan is $O(n)$.

| You want | Stack order (bottom → top) | Pop while |
|---|---|---|
| next **greater** | decreasing | `top < x` |
| next **smaller** | increasing | `top > x` |
| previous greater / smaller | same stacks; read the new top *after* popping | — |

### 1.2.1 [739. Daily Temperatures](https://leetcode.com/problems/daily-temperatures/) — ★★ · Medium

**Problem.** For each day, how many days until a warmer temperature (0 if never)?  
**Signal.** "Next greater element" → monotonic decreasing stack of indices.  
*New here: The monotonic stack template itself.*

**Fine print.** Ask before coding:
- Strictly warmer, or equal? → Strictly warmer.
- No warmer day later? → Write 0.

> 💡 **Key insight.** Days still waiting for a warmer day form a decreasing sequence; a warmer day answers all the cooler ones on top at once.

**Brute force →** For each day, scan forward → O(n²). The stack answers every day in O(n) total.

**Steps**
1. Keep a stack of indices still waiting.
2. For day `i`, while the top day is cooler, pop it and record `i − j`.
3. Push `i`.
4. Days left on the stack never get warmer (answer 0).

**Complexity**

- **Time:** O(n) — each index pushed and popped once.
- **Space:** O(n).

In [18]:
def daily_temperatures(temperatures: list[int]) -> list[int]:
    """Days to wait for a warmer temperature."""
    out = [0] * len(temperatures)
    # indices still waiting; temperatures decrease toward the top
    stack: list[int] = []
    for i, t in enumerate(temperatures):
        # today is the warmer day for every cooler waiting day
        while stack and temperatures[stack[-1]] < t:
            j = stack.pop()
            out[j] = i - j
        stack.append(i)
    return out


assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
assert daily_temperatures([30, 60, 90]) == [1, 1, 0]
print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))

[1, 1, 4, 2, 1, 1, 0, 0]


### 1.2.2 [496. Next Greater Element I](https://leetcode.com/problems/next-greater-element-i/) — ★ · Easy

**Problem.** For each value in `nums1` (a subset of `nums2`), the next greater value to its right in `nums2`, or −1.  
**Signal.** Precompute next-greater for all of `nums2` with a monotonic stack, then look up.  
*New here: Precompute all answers once, then answer queries from a map.*

**Fine print.** Ask before coding:
- Are all values unique? → Yes, so a value-to-answer map works.
- Is `nums1` a subset of `nums2`? → Yes.

> 💡 **Key insight.** The next greater element depends only on `nums2`, so compute it for every value once and look the queries up.

**Brute force →** For each query, find it in `nums2` and scan right → O(m · n). Precompute + map → O(m + n).

**Steps**
1. Run the monotonic stack over `nums2` (values are distinct).
2. Each pop records `next_greater[popped] = x`.
3. Answer each query with `next_greater.get(x, −1)`.

**Complexity**

- **Time:** O(m + n).
- **Space:** O(n).

In [19]:
def next_greater_element(nums1: list[int], nums2: list[int]) -> list[int]:
    """Next greater value in nums2 for each query value."""
    nxt: dict[int, int] = {}
    stack: list[int] = []
    for x in nums2:
        # x answers every smaller value waiting on the stack
        while stack and stack[-1] < x:
            nxt[stack.pop()] = x
        stack.append(x)
    return [nxt.get(x, -1) for x in nums1]


assert next_greater_element([4, 1, 2], [1, 3, 4, 2]) == [-1, 3, -1]
assert next_greater_element([2, 4], [1, 2, 3, 4]) == [3, -1]
print(next_greater_element([4, 1, 2], [1, 3, 4, 2]))

[-1, 3, -1]


### 1.2.3 [503. Next Greater Element II](https://leetcode.com/problems/next-greater-element-ii/) — ★ · Medium

**Problem.** Next greater element in a *circular* array.  
**Signal.** Circular → iterate over the array twice (indices `i % n`).  
*New here: Handling wrap-around by looping twice.*

**Fine print.** Ask before coding:
- Circular, so can you wrap around once? → Yes, which is why you loop over `2n`.
- Duplicates? → Yes, so strictly greater only.

> 💡 **Key insight.** Walking the array twice lets every element see all elements after it *and* the ones before it (wrapped around).

**Brute force →** For each index, scan up to n − 1 next positions circularly → O(n²). Two passes of the stack → O(n).

**Steps**
1. Loop `i` from 0 to `2n − 1`, using `nums[i % n]`.
2. Pop and answer smaller waiting indices as usual.
3. Push indices only during the first pass.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [20]:
def next_greater_elements(nums: list[int]) -> list[int]:
    """Circular next greater element."""
    n = len(nums)
    out = [-1] * n
    stack: list[int] = []
    # second lap lets elements see the ones before them
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            out[stack.pop()] = x
        # each index waits only once
        if i < n:
            stack.append(i)
    return out


assert next_greater_elements([1, 2, 1]) == [2, -1, 2]
assert next_greater_elements([1, 2, 3, 4, 3]) == [2, 3, 4, -1, 4]
print(next_greater_elements([1, 2, 1]))

[2, -1, 2]


### 1.2.4 [901. Online Stock Span](https://leetcode.com/problems/online-stock-span/) — ★ · Medium

**Problem.** For each new price, how many consecutive days (including today) had price ≤ today?  
**Signal.** Previous greater element, online → monotonic stack of (price, span).  
*New here: Online version: absorbing popped entries' spans into the new one.*

**Fine print.** Ask before coding:
- Is the span counted with `<=`? → Yes, and today counts too.
- Online, one `next` call per price? → Yes, prices cannot be preprocessed.

> 💡 **Key insight.** Once today's price beats an earlier day, that day (and everything it covered) is covered by today forever, so its span can be absorbed.

**Brute force →** Walk back day by day for each price → O(n) per call. The stack is amortised O(1).

**Steps**
1. Start today's span at 1.
2. While the top's price ≤ today's, pop it and add its span.
3. Push `(price, span)` and return the span.

**Complexity**

- **Time:** Amortised O(1) per call.
- **Space:** O(n).

In [21]:
class StockSpanner:
    """Online stock span."""

    def __init__(self) -> None:
        # (price, days covered by it); prices decrease toward the top
        self.stack: list[tuple[int, int]] = []

    def next(self, price: int) -> int:
        span = 1
        # absorb every earlier day not above today
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


sp = StockSpanner()
out = [sp.next(p) for p in [100, 80, 60, 70, 60, 75, 85]]
assert out == [1, 1, 1, 2, 1, 4, 6]
print(out)

[1, 1, 1, 2, 1, 4, 6]


### 1.2.5 [853. Car Fleet](https://leetcode.com/problems/car-fleet/) — ★★ · Medium

**Problem.** Cars at `position[i]` with `speed[i]` drive to `target`; a faster car catching a slower one joins it. How many fleets arrive?  
**Signal.** Sort by position descending, compare arrival times → stack of fleet times.  
*New here: Sorting first, then a monotonic stack over derived values (arrival times).*

**Fine print.** Ask before coding:
- Are positions unique? → Yes.
- Does a car that catches up exactly at the target count as one fleet? → Yes.
- Can cars pass each other? → No.

> 💡 **Key insight.** Processing from the car nearest the target, a car whose solo arrival time is not later than the fleet ahead catches it and merges.

**Brute force →** Simulate time steps → depends on distances, slow. Sort + arrival times → O(n log n).

**Steps**
1. Sort cars by position, nearest to target first.
2. Arrival time = `(target − pos) / speed`.
3. If the time is greater than the last fleet's time, it is a new fleet; otherwise it merges.
4. Count fleets.

**Complexity**

- **Time:** O(n log n) — the sort.
- **Space:** O(n).

In [22]:
def car_fleet(target: int, position: list[int], speed: list[int]) -> int:
    """Number of fleets reaching the target."""
    fleets: list[float] = []
    # closest car to the target first
    for pos, spd in sorted(zip(position, speed), reverse=True):
        time = (target - pos) / spd
        # slower than the fleet ahead: it can never catch up, new fleet
        if not fleets or time > fleets[-1]:
            fleets.append(time)
    return len(fleets)


assert car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
assert car_fleet(10, [3], [3]) == 1
assert car_fleet(100, [0, 2, 4], [4, 2, 1]) == 1
print(car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))

3


### 1.2.6 [84. Largest Rectangle in Histogram](https://leetcode.com/problems/largest-rectangle-in-histogram/) — ★★ · Hard

**Problem.** Largest rectangle area in a histogram.  
**Signal.** Each bar's widest rectangle extends to the previous and next *smaller* bars → monotonic increasing stack.  
*New here: Using *both* neighbours from one pop: the popper is the right boundary, the new top the left.*

**Fine print.** Ask before coding:
- Is every bar width 1? → Yes.
- Can heights be 0? → Yes.

> 💡 **Key insight.** The tallest rectangle using bar `j` as its height spans from the previous smaller bar to the next smaller bar; popping `j` reveals both.

**Brute force →** For each bar, expand left and right while bars are at least as tall → O(n²). The stack is O(n).

**Steps**
1. Keep indices with increasing heights; append a 0 sentinel to flush at the end.
2. When bar `i` is lower than the top, pop the top `j` (its next smaller is `i`).
3. Its previous smaller is the new top (or −1); width = `i − left − 1`.
4. Update the best area with `height[j] * width`; push `i`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

⚠️ **Trap.** Forgetting the 0 sentinel leaves bars on the stack that are never measured.

In [23]:
def largest_rectangle_area(heights: list[int]) -> int:
    """Largest rectangle in a histogram."""
    # indices with increasing heights
    stack: list[int] = []
    best = 0
    # the 0-height sentinel flushes every remaining bar
    for i, h in enumerate(heights + [0]):
        while stack and heights[stack[-1]] >= h:
            # i is the first lower bar to the right of the popped bar
            height = heights[stack.pop()]
            # the new top is the first lower bar to its left
            left = stack[-1] if stack else -1
            best = max(best, height * (i - left - 1))
        stack.append(i)
    return best


assert largest_rectangle_area([2, 1, 5, 6, 2, 3]) == 10
assert largest_rectangle_area([2, 4]) == 4
assert largest_rectangle_area([1, 1, 1, 1]) == 4
print(largest_rectangle_area([2, 1, 5, 6, 2, 3]))

10


### 1.2.7 [85. Maximal Rectangle](https://leetcode.com/problems/maximal-rectangle/) — ★ · Hard

**Problem.** Largest rectangle of 1s in a binary matrix.  
**Signal.** Each row turns into a histogram of consecutive 1s above → run Largest Rectangle per row.  
*New here: Reducing a 2D problem to repeated 1D calls.*

**Fine print.** Ask before coding:
- Are cells the characters `"1"` and `"0"`, or ints? → Characters on LeetCode.
- Rectangle, not square? → Yes, a rectangle.

> 💡 **Key insight.** If the rectangle's bottom is on row `r`, it lives in the histogram of consecutive 1s ending at row `r`.

**Brute force →** Every pair of corners → O(rows² · cols²). Histogram per row → O(rows · cols).

**Steps**
1. `heights[c]` = consecutive `"1"`s ending at the current row in column `c`.
2. Update `heights` row by row (reset to 0 on `"0"`).
3. Run Largest Rectangle in Histogram on each row's heights.
4. Keep the best.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(cols).

In [24]:
def maximal_rectangle(matrix: list[list[str]]) -> int:
    """Largest all-ones rectangle area."""
    heights = [0] * len(matrix[0])
    best = 0
    for row in matrix:
        # histogram of 1-runs ending at this row
        for c, cell in enumerate(row):
            heights[c] = heights[c] + 1 if cell == "1" else 0
        # best rectangle whose bottom edge is this row
        best = max(best, largest_rectangle_area(heights))
    return best


grid = [list("10100"), list("10111"), list("11111"), list("10010")]
assert maximal_rectangle(grid) == 6
assert maximal_rectangle([["0"]]) == 0
print(maximal_rectangle(grid))

6


### 1.2.8 [402. Remove K Digits](https://leetcode.com/problems/remove-k-digits/) — ★ · Medium

**Problem.** Remove `k` digits from a number string to make it as small as possible.  
**Signal.** Lexicographically smallest → greedy monotonic increasing stack.  
*New here: Greedy monotonic stack with a removal budget.*

**Fine print.** Ask before coding:
- Should leading zeros be removed from the result? → Yes.
- If everything is removed? → Return `"0"`.
- Is the input free of leading zeros? → Yes, unless it is `"0"`.

> 💡 **Key insight.** A digit followed by a smaller one should go first — fixing an earlier position matters more than any later one.

**Brute force →** Try all `C(n, k)` deletions → exponential. Greedy stack → O(n).

**Steps**
1. For each digit, while `k > 0` and the top is larger, pop it and decrement `k`.
2. Push the digit.
3. If `k` remains, drop digits from the end (the stack is increasing).
4. Strip leading zeros; empty → `"0"`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [25]:
def remove_kdigits(num: str, k: int) -> str:
    """Smallest number after deleting k digits."""
    stack: list[str] = []
    for d in num:
        # a larger digit before a smaller one is worth deleting
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    # remaining deletions come off the (largest) end
    if k:
        stack = stack[:-k]
    return "".join(stack).lstrip("0") or "0"


assert remove_kdigits("1432219", 3) == "1219"
assert remove_kdigits("10200", 1) == "200"
assert remove_kdigits("10", 2) == "0"
print(remove_kdigits("1432219", 3))

1219


### 1.2.9 [316. Remove Duplicate Letters](https://leetcode.com/problems/remove-duplicate-letters/) — ★ · Medium

**Problem.** Remove duplicate letters so each appears once, giving the lexicographically smallest result.  
**Signal.** Remove K Digits plus "only pop a letter if it appears again later".  
*New here: A pop condition that checks the future (last occurrence).*

**Fine print.** Ask before coding:
- Must every letter appear exactly once? → Yes, each distinct letter once.
- Must the relative order of the input be kept? → Yes, it is a subsequence.

> 💡 **Key insight.** A larger letter on top can be removed only if a later copy exists to put it back.

**Brute force →** Generate every subsequence with each letter once → exponential. Greedy stack → O(n).

**Steps**
1. Record each letter's last index.
2. Skip a letter already in the stack.
3. Pop larger letters from the top while they occur again later.
4. Push the letter and mark it used.

**Complexity**

- **Time:** O(n).
- **Space:** O(26).

In [26]:
def remove_duplicate_letters(s: str) -> str:
    """Smallest subsequence containing each letter once."""
    last = {ch: i for i, ch in enumerate(s)}
    stack: list[str] = []
    used: set[str] = set()
    for i, ch in enumerate(s):
        # already placed at its best position
        if ch in used:
            continue
        # drop a larger letter if it can be re-added later
        while stack and stack[-1] > ch and last[stack[-1]] > i:
            used.remove(stack.pop())
        stack.append(ch)
        used.add(ch)
    return "".join(stack)


assert remove_duplicate_letters("bcabc") == "abc"
assert remove_duplicate_letters("cbacdcbc") == "acdb"
print(remove_duplicate_letters("cbacdcbc"))

acdb


### 1.2.10 [907. Sum of Subarray Minimums](https://leetcode.com/problems/sum-of-subarray-minimums/) — ★ · Medium

**Problem.** Sum of `min(subarray)` over all subarrays, mod 1e9+7.  
**Signal.** Contribution technique: count subarrays where each element is the minimum.  
*New here: The *contribution technique* — sum over elements instead of over subarrays.*

**Fine print.** Ask before coding:
- Is the answer taken modulo 1e9 + 7? → Yes.
- Duplicates? → Yes, so ties need careful strict and non-strict comparisons.

> 💡 **Key insight.** `arr[j]` is the minimum of exactly `left × right` subarrays, where `left` and `right` are the distances to the previous smaller and next smaller-or-equal elements.

**Brute force →** Enumerate subarrays with running minimums → O(n²). Contributions via one stack pass → O(n).

**Steps**
1. Monotonic increasing stack over `arr + [0]` (sentinel flushes everything).
2. When `x` pops `j`, `i` is `j`'s next smaller-or-equal, the new top its previous smaller.
3. Add `arr[j] * (j − prev) * (i − j)`.
4. Take the total mod 1e9+7.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

⚠️ **Trap.** Ties must be strict on one side and non-strict on the other, or equal minimums are double-counted.

In [27]:
def sum_subarray_mins(arr: list[int]) -> int:
    """Sum of minimums of all subarrays mod 1e9+7."""
    total = 0
    stack: list[int] = []
    # trailing 0 forces every index to be popped
    extended = arr + [0]
    for i, x in enumerate(extended):
        while stack and extended[stack[-1]] >= x:
            j = stack.pop()
            # choices for the left end: back to the previous smaller element
            left = j - (stack[-1] if stack else -1)
            # choices for the right end: up to the next smaller-or-equal element
            right = i - j
            total += arr[j] * left * right
        stack.append(i)
    return total % (10**9 + 7)


assert sum_subarray_mins([3, 1, 2, 4]) == 17
assert sum_subarray_mins([11, 81, 94, 43, 3]) == 444
print(sum_subarray_mins([3, 1, 2, 4]))

17


### 1.2.11 [1762. Buildings With an Ocean View](https://leetcode.com/problems/buildings-with-an-ocean-view/) — ★ · Medium · Premium

**Problem.** Ocean on the right; return indices of buildings strictly taller than everything to their right.  
**Signal.** Scan from the right with a running max (or a decreasing stack from the left).  
*New here: When only one side matters, a running maximum replaces the stack.*

**Fine print.** Ask before coding:
- Strictly taller to have a view? → Yes.
- Return indices in increasing order? → Yes.

> 💡 **Key insight.** A building sees the ocean iff it is taller than the tallest building to its right — a single running max from the right.

**Brute force →** For each building, scan everything to its right → O(n²). A right-to-left running max → O(n).

**Steps**
1. Walk from the right, keeping `tallest`.
2. A building taller than `tallest` has a view; record it and update `tallest`.
3. Reverse the recorded indices.

**Complexity**

- **Time:** O(n).
- **Space:** O(1) extra besides the output.

In [28]:
def find_buildings(heights: list[int]) -> list[int]:
    """Indices of buildings that see the ocean."""
    out, tallest = [], 0
    for i in range(len(heights) - 1, -1, -1):
        # taller than everything to its right
        if heights[i] > tallest:
            out.append(i)
            tallest = heights[i]
    return out[::-1]


assert find_buildings([4, 2, 3, 1]) == [0, 2, 3]
assert find_buildings([4, 3, 2, 1]) == [0, 1, 2, 3]
assert find_buildings([1, 3, 2, 4]) == [3]
print(find_buildings([4, 2, 3, 1]))

[0, 2, 3]


---
# 2) Binary Search

**In plain English.** If a yes/no question flips from "no" to "yes" exactly once as you
move along a range, you can find the flip point by always checking the middle and throwing
away the half that cannot contain it.

> **Analogy.** Guessing a number between 1 and 100 with "higher / lower" hints: seven
> guesses are always enough, because each guess halves what is left.

One template avoids every off-by-one: find the **first index where a monotone predicate
becomes true**.

```python
lo, hi = 0, n            # answer lies in [lo, hi]; hi = n means "never true"
while lo < hi:
    mid = (lo + hi) // 2
    if predicate(mid):
        hi = mid         # mid may be the answer: keep it
    else:
        lo = mid + 1     # mid is definitely not the answer: drop it
return lo
```

**Search on the answer.** For "the minimum speed / capacity / day such that something is
feasible", the answer space is a range of integers and feasibility is monotone (if speed
`v` works, any faster speed works). Binary-search that range and test feasibility with an
$O(n)$ greedy check.

**Recognition signals**

- "sorted array", "rotated sorted", "O(log n)"
- "first / last position", "insert position", "smallest ≥ x"
- "minimum X such that …", "maximum X such that …", "at least / at most k days / hours"
- the input is huge (≥ 10⁹) but a single check is cheap
- "k-th smallest" in a structure where you can *count* elements ≤ v

**Common bugs**

- Mixing templates (`lo <= hi` with `hi = mid`) → infinite loop. Pick one and stick to it.
- Wrong search range: the answer must be inside `[lo, hi]` at the start (e.g. capacity
  starts at `max(weights)`, not 1).
- A predicate that is not monotone — then binary search silently returns garbage.
- Integer overflow in `(lo + hi) / 2` in other languages (use `lo + (hi − lo) // 2`).
- Ceil division: `(a + b − 1) // b`, not `a // b + 1`.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Two pointers | discards one element per step; binary search discards half |
| Sliding window | optimises a contiguous block; binary search on the answer optimises a *parameter* |
| Greedy | often the *feasibility check* inside binary search on the answer |
| Heap for k-th smallest | O(k log n); counting + binary search on value is better for sorted matrices |

**Learning path.** Exact search → lower bound / insert position → first true predicate
(First Bad Version) → both bounds → 2D as 1D → answer-space search (Sqrt) → searches on
slopes and parity (Peak, Single Element) → windows (K Closest). Then rotated arrays
(which half is sorted?), and finally search on the answer: Koko → shipping → split
array → counting-based k-th smallest → partition search (Median) → timestamps.

## 2.1 Classic binary search

Every question here uses the first-true template; what changes is the predicate. Write the
predicate as a sentence ("`nums[i] >= target`", "`is_bad(v)`", "`mid * mid > x`") before
writing the loop.

### 2.1.1 [704. Binary Search](https://leetcode.com/problems/binary-search/) — ★★ · Easy

**Problem.** Index of `target` in a sorted array, or −1.  
**Signal.** Sorted + lookup → binary search.  
*New here: The first-true template with predicate `nums[i] >= target`.*

**Fine print.** Ask before coding:
- Sorted ascending with distinct values? → Yes.
- Not found? → Return −1. Must it be O(log n)? → Yes.

> 💡 **Key insight.** The first index with `nums[i] >= target` is where `target` would be; it is there iff that slot equals `target`.

**Brute force →** Linear scan → O(n). Binary search → O(log n).

**Steps**
1. `lo = 0`, `hi = n`.
2. If `nums[mid] >= target`, keep `mid` (`hi = mid`); else drop it (`lo = mid + 1`).
3. Stop when `lo == hi`.
4. Return `lo` if it is in range and holds `target`, else −1.

**Complexity**

- **Time:** O(log n) — the range halves each step.
- **Space:** O(1).

In [29]:
def search(nums: list[int], target: int) -> int:
    """Index of target in sorted nums, or -1."""
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        # mid could be the first index >= target
        if nums[mid] >= target:
            hi = mid
        # mid is too small: the answer is to its right
        else:
            lo = mid + 1
    # lo is the lower bound; check it actually holds target
    return lo if lo < len(nums) and nums[lo] == target else -1


assert search([-1, 0, 3, 5, 9, 12], 9) == 4
assert search([-1, 0, 3, 5, 9, 12], 2) == -1
print(search([-1, 0, 3, 5, 9, 12], 9))

4


### 2.1.2 [35. Search Insert Position](https://leetcode.com/problems/search-insert-position/) — ★★ · Easy

**Problem.** Index of `target`, or where it would be inserted to keep order.  
**Signal.** The template's answer *is* the insertion point (lower bound).  
*New here: Seeing that the template's output is meaningful even when the target is absent.*

**Fine print.** Ask before coding:
- Distinct values? → Yes.
- Insert position can be `n`? → Yes, past the end.

> 💡 **Key insight.** The first index with `nums[i] >= target` is exactly where `target` goes to keep the array sorted.

**Brute force →** Scan for the first element ≥ target → O(n). The lower bound is O(log n) (`bisect_left`).

**Steps**
1. Run the first-true template with `nums[mid] >= target`.
2. Return `lo` (may equal `n`: insert at the end).

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [30]:
def search_insert(nums: list[int], target: int) -> int:
    """Lower bound of target in nums."""
    # hi = len(nums): inserting after everything is allowed
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert search_insert([1, 3, 5, 6], 5) == 2
assert search_insert([1, 3, 5, 6], 2) == 1
assert search_insert([1, 3, 5, 6], 7) == 4
print(search_insert([1, 3, 5, 6], 2))

1


### 2.1.3 [278. First Bad Version](https://leetcode.com/problems/first-bad-version/) — ★★ · Easy

**Problem.** Versions `1..n`; after the first bad one, all are bad. Find it with few API calls.  
**Signal.** Monotone predicate `is_bad(v)` → first-true binary search.  
*New here: The predicate is a black-box function, not an array comparison.*

**Fine print.** Ask before coding:
- Is the goal to minimise API calls? → Yes.
- Is `n` up to 2³¹ − 1? → Yes; mid overflow matters in fixed-width languages.

> 💡 **Key insight.** "Good, good, …, bad, bad" is a monotone predicate, so the template finds the first bad version directly.

**Brute force →** Check versions in order → O(n) calls. Binary search → O(log n) calls.

**Steps**
1. Range `[1, n]`.
2. `is_bad(mid)` → `hi = mid`; else `lo = mid + 1`.
3. Return `lo`.

**Complexity**

- **Time:** O(log n) API calls.
- **Space:** O(1).

In [31]:
def first_bad_version(n: int, is_bad) -> int:
    """First version for which is_bad is True."""
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        # mid is bad: the first bad one is mid or earlier
        if is_bad(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


assert first_bad_version(5, lambda v: v >= 4) == 4
assert first_bad_version(1, lambda v: v >= 1) == 1
print(first_bad_version(2**31 - 1, lambda v: v >= 1702766719))

1702766719


### 2.1.4 [34. Find First and Last Position of Element in Sorted Array](https://leetcode.com/problems/find-first-and-last-position-of-element-in-sorted-array/) — ★★ · Medium

**Problem.** `[first, last]` index of `target` in a sorted array, or `[-1, -1]`.  
**Signal.** Two binary searches: lower bound of `target` and lower bound of `target + 1`.  
*New here: Getting the *last* position from a lower bound of the next value.*

**Fine print.** Ask before coding:
- Must it be O(log n)? → Yes, two binary searches.
- Not found? → Return `[-1, -1]`.

> 💡 **Key insight.** The last copy of `target` sits just before the first element greater than `target`, i.e. `lower_bound(target + 1) − 1`.

**Brute force →** Find one copy, then expand left and right → O(n) when all values are equal. Two lower bounds → O(log n).

**Steps**
1. `first = lower_bound(target)`.
2. If `first` is out of range or not `target`, return `[-1, -1]`.
3. `last = lower_bound(target + 1) − 1`.

**Complexity**

- **Time:** O(log n) — two searches.
- **Space:** O(1).

In [32]:
def search_range(nums: list[int], target: int) -> list[int]:
    """First and last index of target."""

    def lower_bound(x: int) -> int:
        # first index with nums[i] >= x
        lo, hi = 0, len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] >= x:
                hi = mid
            else:
                lo = mid + 1
        return lo

    first = lower_bound(target)
    if first == len(nums) or nums[first] != target:
        return [-1, -1]
    # last copy is just before the first element > target
    return [first, lower_bound(target + 1) - 1]


assert search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert search_range([], 0) == [-1, -1]
print(search_range([5, 7, 7, 8, 8, 10], 8))

[3, 4]


### 2.1.5 [74. Search a 2D Matrix](https://leetcode.com/problems/search-a-2d-matrix/) — ★★ · Medium

**Problem.** Rows sorted, and each row starts after the previous row ends. Is `target` present?  
**Signal.** The matrix is one sorted array of length `m·n` in disguise.  
*New here: Mapping a 1D index onto 2D coordinates.*

**Fine print.** Ask before coding:
- Does each row start above the previous row's last value? → Yes, so the matrix reads as one sorted array.
- Can it be a single row or column? → Yes.

> 💡 **Key insight.** Read row by row, the matrix is a single sorted list; index `k` is cell `(k // cols, k % cols)`.

**Brute force →** Scan everything → O(mn). Binary search per row → O(m log n). One search over `m·n` → O(log mn).

**Steps**
1. Search `k` in `[0, m·n − 1]`.
2. Read `matrix[k // cols][k % cols]`.
3. Equal → found; smaller → go right; larger → go left.

**Complexity**

- **Time:** O(log(mn)).
- **Space:** O(1).

In [33]:
def search_matrix(matrix: list[list[int]], target: int) -> bool:
    """True if target is in the row-major sorted matrix."""
    rows, cols = len(matrix), len(matrix[0])
    lo, hi = 0, rows * cols - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        # flat index -> (row, column)
        value = matrix[mid // cols][mid % cols]
        if value == target:
            return True
        if value < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False


m = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert search_matrix(m, 3)
assert not search_matrix(m, 13)
print(search_matrix(m, 34))

True


### 2.1.6 [240. Search a 2D Matrix II](https://leetcode.com/problems/search-a-2d-matrix-ii/) — ★ · Medium

**Problem.** Rows and columns sorted independently. Is `target` present?  
**Signal.** Start at the top-right corner: it splits the remaining matrix like a BST root.  
*New here: A "staircase" walk — not binary search, but the same discard idea.*

**Fine print.** Ask before coding:
- Rows and columns sorted, but not as one array? → Yes, so start the staircase from the top-right.
- Duplicates? → Possible; it does not matter.

> 💡 **Key insight.** At the top-right corner, everything below is larger and everything to the left is smaller, so one comparison eliminates a whole row or column.

**Brute force →** Binary search each row → O(m log n). The staircase → O(m + n).

**Steps**
1. Start at the top-right cell.
2. Equal → found.
3. Too big → move left (this column is all too big below).
4. Too small → move down (this row is all too small to the left).

**Complexity**

- **Time:** O(m + n) — each step removes a row or a column.
- **Space:** O(1).

In [34]:
def search_matrix_ii(matrix: list[list[int]], target: int) -> bool:
    """True if target is in a row- and column-sorted matrix."""
    # top-right corner: left is smaller, down is larger
    r, c = 0, len(matrix[0]) - 1
    while r < len(matrix) and c >= 0:
        value = matrix[r][c]
        if value == target:
            return True
        # this column is too big from here down
        if value > target:
            c -= 1
        # this row is too small from here left
        else:
            r += 1
    return False


m = [
    [1, 4, 7, 11, 15],
    [2, 5, 8, 12, 19],
    [3, 6, 9, 16, 22],
    [10, 13, 14, 17, 24],
    [18, 21, 23, 26, 30],
]
assert search_matrix_ii(m, 5)
assert not search_matrix_ii(m, 20)
print(search_matrix_ii(m, 5))

True


### 2.1.7 [69. Sqrt(x)](https://leetcode.com/problems/sqrtx/) — ★★ · Easy

**Problem.** Integer square root (floor), no built-in power functions.  
**Signal.** Largest `r` with `r² ≤ x` → binary search on the answer.  
*New here: Searching over *answers* (integers), not array indices.*

**Fine print.** Ask before coding:
- Round down? → Yes, floor.
- Can `x` be 0? → Yes.
- Is `x` up to 2³¹ − 1? → Yes, so `mid * mid` overflows 32 bits in other languages.

> 💡 **Key insight.** "Largest r with r² ≤ x" = "first r with r² > x" minus one, and `r² > x` is monotone.

**Brute force →** Try `r = 0, 1, 2, …` → O(√x). Binary search → O(log x).

**Steps**
1. Range `[0, x + 1]`.
2. Predicate `mid * mid > x`.
3. Return `lo − 1`.

**Complexity**

- **Time:** O(log x).
- **Space:** O(1).

In [35]:
def my_sqrt(x: int) -> int:
    """floor(sqrt(x))."""
    lo, hi = 0, x + 1
    while lo < hi:
        mid = (lo + hi) // 2
        # first r whose square overshoots x
        if mid * mid > x:
            hi = mid
        else:
            lo = mid + 1
    # step back to the last r with r*r <= x
    return lo - 1


assert my_sqrt(4) == 2
assert my_sqrt(8) == 2
assert my_sqrt(0) == 0
assert my_sqrt(2147395599) == 46339
print(my_sqrt(8))

2


### 2.1.8 [367. Valid Perfect Square](https://leetcode.com/problems/valid-perfect-square/) — ★ · Easy

**Problem.** Is `num` a perfect square (no `sqrt`)?  
**Signal.** Same search as Sqrt(x), then check equality.  
*New here: Reusing a search result for a yes/no answer.*

**Fine print.** Ask before coding:
- Is `num ≥ 1`? → Yes.
- No `sqrt` or `**`? → Correct, use binary search.

> 💡 **Key insight.** `num` is a perfect square iff its integer square root squared gives it back.

**Brute force →** Try every `r` up to √num → O(√n). Binary search → O(log n).

**Steps**
1. `r = my_sqrt(num)`.
2. Return `r * r == num`.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [36]:
def is_perfect_square(num: int) -> bool:
    """True if num is a perfect square."""
    r = my_sqrt(num)
    # the floor root squares back exactly only for perfect squares
    return r * r == num


assert is_perfect_square(16)
assert not is_perfect_square(14)
print(is_perfect_square(808201))

True


### 2.1.9 [162. Find Peak Element](https://leetcode.com/problems/find-peak-element/) — ★★ · Medium

**Problem.** Return any index whose value is greater than its neighbours (`nums[-1] = nums[n] = −∞`).  
**Signal.** Binary search on the slope: go uphill.  
*New here: Binary search on an *unsorted* array, using a local slope as the predicate.*

**Fine print.** Ask before coding:
- Are neighbours never equal? → Yes, which is what guarantees a peak exists.
- Several peaks? → Return any one.

> 💡 **Key insight.** If the array rises at `mid`, a peak must exist to the right (it either keeps rising to the edge, which counts, or turns down).

**Brute force →** Scan for any `nums[i] > nums[i + 1]` → O(n). Following the slope → O(log n).

**Steps**
1. Range `[0, n − 1]`.
2. `nums[mid] < nums[mid + 1]` → peak is right of `mid`: `lo = mid + 1`.
3. Otherwise a peak is at `mid` or left: `hi = mid`.
4. Return `lo`.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [37]:
def find_peak_element(nums: list[int]) -> int:
    """Index of some local maximum."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # rising slope: walk uphill to the right
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        # falling slope: mid or something left of it is a peak
        else:
            hi = mid
    return lo


assert find_peak_element([1, 2, 3, 1]) == 2
assert find_peak_element([1, 2, 1, 3, 5, 6, 4]) in (1, 5)
print(find_peak_element([1, 2, 1, 3, 5, 6, 4]))

5


### 2.1.10 [540. Single Element in a Sorted Array](https://leetcode.com/problems/single-element-in-a-sorted-array/) — ★ · Medium

**Problem.** Sorted array where every element appears twice except one; find it in O(log n).  
**Signal.** Before the single element, pairs start at even indices; after it, at odd indices.  
*New here: A predicate based on index parity.*

**Fine print.** Ask before coding:
- Exactly one single element? → Yes.
- Is O(log n) required? → Yes; XOR gives O(n).

> 💡 **Key insight.** The single element shifts every later pair by one, so "pair starts at an even index" is true before it and false after it.

**Brute force →** XOR everything or scan pairs → O(n). Parity binary search → O(log n).

**Steps**
1. Make `mid` even (subtract 1 if odd).
2. If `nums[mid] == nums[mid + 1]`, pairs are intact up to here → `lo = mid + 2`.
3. Otherwise the single element is at `mid` or left → `hi = mid`.
4. Return `nums[lo]`.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [38]:
def single_non_duplicate(nums: list[int]) -> int:
    """The element appearing once."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # look at the start of a pair (even index)
        if mid % 2:
            mid -= 1
        # pair intact: the single one is further right
        if nums[mid] == nums[mid + 1]:
            lo = mid + 2
        # pairing broken at or before mid
        else:
            hi = mid
    return nums[lo]


assert single_non_duplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2
assert single_non_duplicate([3, 3, 7, 7, 10, 11, 11]) == 10
print(single_non_duplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]))

2


### 2.1.11 [1539. Kth Missing Positive Number](https://leetcode.com/problems/kth-missing-positive-number/) — ★ · Easy

**Problem.** Sorted positive integers; return the `k`-th positive integer missing from the array.  
**Signal.** Missing count before index `i` is `arr[i] − (i + 1)`, which is monotone.  
*New here: Inventing a derived monotone quantity to search on.*

**Fine print.** Ask before coding:
- Strictly increasing positive integers? → Yes.
- Can the answer be past the last element? → Yes.

> 💡 **Key insight.** Without gaps, `arr[i]` would be `i + 1`; the difference counts how many numbers are missing before `arr[i]`, and it only grows.

**Brute force →** Walk the integers counting misses → O(n + k). Binary search on the missing count → O(log n).

**Steps**
1. Find the first index `lo` with `arr[lo] − (lo + 1) ≥ k`.
2. The `lo` present numbers before it push the answer up by `lo`.
3. Return `lo + k`.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [39]:
def find_kth_positive(arr: list[int], k: int) -> int:
    """k-th missing positive integer."""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        # numbers missing before arr[mid]
        if arr[mid] - (mid + 1) >= k:
            hi = mid
        else:
            lo = mid + 1
    # k-th missing, shifted by the lo present numbers before it
    return lo + k


assert find_kth_positive([2, 3, 4, 7, 11], 5) == 9
assert find_kth_positive([1, 2, 3, 4], 2) == 6
print(find_kth_positive([2, 3, 4, 7, 11], 5))

9


### 2.1.12 [658. Find K Closest Elements](https://leetcode.com/problems/find-k-closest-elements/) — ★★ · Medium

**Problem.** The `k` values closest to `x` in a sorted array, returned sorted (ties → smaller).  
**Signal.** The answer is a window of length `k` → binary search the window's left edge.  
*New here: Binary searching the *start of a window*.*

**Fine print.** Ask before coding:
- How are distance ties broken? → The smaller value wins.
- Is `k ≤ n`? → Yes. Should the output be sorted? → Yes, ascending.

> 💡 **Key insight.** Comparing the window's left end with the element just past its right end tells you which way the best window lies.

**Brute force →** Sort by distance → O(n log n). Two pointers shrinking from both ends → O(n − k). Binary search the left edge → O(log(n − k) + k).

**Steps**
1. Search `left` in `[0, n − k]`.
2. If `x − arr[mid] > arr[mid + k] − x`, the left end is worse → shift right.
3. Otherwise keep `mid` (ties prefer the left).
4. Return `arr[lo : lo + k]`.

**Complexity**

- **Time:** O(log(n − k) + k).
- **Space:** O(1) besides the output.

In [40]:
def find_closest_elements(arr: list[int], k: int, x: int) -> list[int]:
    """k closest values to x, sorted."""
    # candidate window starts
    lo, hi = 0, len(arr) - k
    while lo < hi:
        mid = (lo + hi) // 2
        # left end farther than the next element on the right: slide right
        if x - arr[mid] > arr[mid + k] - x:
            lo = mid + 1
        else:
            hi = mid
    return arr[lo : lo + k]


assert find_closest_elements([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert find_closest_elements([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
print(find_closest_elements([1, 2, 3, 4, 5], 4, 3))

[1, 2, 3, 4]


### 2.1.13 [2300. Successful Pairs of Spells and Potions](https://leetcode.com/problems/successful-pairs-of-spells-and-potions/) — ★ · Medium

**Problem.** For each spell, count potions with `spell × potion ≥ success`.  
**Signal.** Sort potions once; each spell needs a threshold lookup.  
*New here: Sort once, then many binary searches (one per query).*

**Fine print.** Ask before coding:
- Are potions sorted? → No, sort them first.
- Can the product exceed 32 bits? → Yes, up to 10¹⁰; use 64-bit in other languages.

> 💡 **Key insight.** A spell `s` needs potions ≥ `ceil(success / s)`; in sorted potions, those form a suffix.

**Brute force →** Every spell × every potion → O(m · n). Sort + binary search → O((m + n) log m).

**Steps**
1. Sort potions.
2. For each spell, `need = ceil(success / s)`.
3. Lower bound of `need` in potions.
4. Count = `m − lower_bound`.

**Complexity**

- **Time:** O((n + m) log m) — sort plus one search per spell.
- **Space:** O(m) for the sorted copy.

In [41]:
def successful_pairs(spells: list[int], potions: list[int], success: int) -> list[int]:
    """Number of successful potions per spell."""
    potions = sorted(potions)
    m = len(potions)
    out = []
    for s in spells:
        # weakest potion that still works (ceil division)
        need = (success + s - 1) // s
        lo, hi = 0, m
        while lo < hi:
            mid = (lo + hi) // 2
            if potions[mid] >= need:
                hi = mid
            else:
                lo = mid + 1
        # every potion from lo onward works
        out.append(m - lo)
    return out


assert successful_pairs([5, 1, 3], [1, 2, 3, 4, 5], 7) == [4, 0, 3]
assert successful_pairs([3, 1, 2], [8, 5, 8], 16) == [2, 0, 2]
print(successful_pairs([5, 1, 3], [1, 2, 3, 4, 5], 7))

[4, 0, 3]


## 2.2 Rotated arrays

A rotated sorted array is two sorted runs. At any `mid`, at least one side is sorted;
compare `nums[mid]` with an end to find out which, then decide whether the target can be
in that sorted side.

### 2.2.1 [153. Find Minimum in Rotated Sorted Array](https://leetcode.com/problems/find-minimum-in-rotated-sorted-array/) — ★★★ · Medium

**Problem.** Minimum of a rotated sorted array of distinct values, O(log n).  
**Signal.** Compare `mid` with the right end to know which half is sorted.  
*New here: Using the right end as a reference to locate the rotation point.*

**Fine print.** Ask before coding:
- Distinct values? → Yes; duplicates make the worst case O(n).
- Can the array be unrotated? → Yes, which is why you compare with `nums[hi]`.

> 💡 **Key insight.** If `nums[mid] > nums[hi]`, the drop (the minimum) is to the right of `mid`; otherwise `mid..hi` is sorted and the minimum is at `mid` or left.

**Brute force →** `min(nums)` → O(n). Binary search → O(log n).

**Steps**
1. `lo = 0`, `hi = n − 1`.
2. `nums[mid] > nums[hi]` → `lo = mid + 1`.
3. Otherwise → `hi = mid`.
4. Return `nums[lo]`.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

⚠️ **Trap.** Comparing with `nums[lo]` instead of `nums[hi]` fails on an unrotated array.

In [42]:
def find_min(nums: list[int]) -> int:
    """Minimum of a rotated sorted array."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # mid sits in the upper run: the drop is to the right
        if nums[mid] > nums[hi]:
            lo = mid + 1
        # mid..hi is sorted: the minimum is mid or earlier
        else:
            hi = mid
    return nums[lo]


assert find_min([3, 4, 5, 1, 2]) == 1
assert find_min([4, 5, 6, 7, 0, 1, 2]) == 0
assert find_min([11, 13, 15, 17]) == 11
print(find_min([4, 5, 6, 7, 0, 1, 2]))

0


### 2.2.2 [33. Search in Rotated Sorted Array](https://leetcode.com/problems/search-in-rotated-sorted-array/) — ★★★ · Medium

**Problem.** Index of `target` in a rotated sorted array of distinct values, O(log n).  
**Signal.** One half around `mid` is always sorted → check whether `target` lies in that half.  
*New here: Deciding direction from "which half is sorted" plus a range check.*

**Fine print.** Ask before coding:
- Distinct values? → Yes.
- Not found? → Return −1.

> 💡 **Key insight.** One half is always sorted, and for a sorted half you can tell in O(1) whether the target lies inside it.

**Brute force →** Linear scan → O(n). Find the rotation point, then binary search the right run → O(log n) but two searches; one combined search is shorter.

**Steps**
1. If `nums[mid] == target`, return `mid`.
2. If `nums[lo] <= nums[mid]`, the left half is sorted: go left iff `nums[lo] <= target < nums[mid]`.
3. Otherwise the right half is sorted: go right iff `nums[mid] < target <= nums[hi]`.
4. Not found → −1.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [43]:
def search_rotated(nums: list[int], target: int) -> int:
    """Index of target in a rotated sorted array, or -1."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        # left half is sorted
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        # right half is sorted
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1


assert search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert search_rotated([1], 0) == -1
assert search_rotated([3, 1], 1) == 1
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))

4


### 2.2.3 [81. Search in Rotated Sorted Array II](https://leetcode.com/problems/search-in-rotated-sorted-array-ii/) — ★ · Medium

**Problem.** Same as above, but duplicates are allowed; return whether `target` exists.  
**Signal.** Duplicates can hide which half is sorted → when `nums[lo] == nums[mid] == nums[hi]`, shrink both ends.  
*New here: Handling an ambiguous comparison by shrinking, at the cost of worst-case O(n).*

**Fine print.** Ask before coding:
- Return a boolean, not an index? → Yes.
- With duplicates, what is the worst-case time? → O(n), when `nums[lo] == nums[mid] == nums[hi]`.

> 💡 **Key insight.** When the two ends and the middle are equal you cannot tell which side is sorted, but those two end values are not the target, so dropping them is safe.

**Brute force →** Linear scan → O(n). Binary search with the shrink → O(log n) on average, O(n) worst case.

**Steps**
1. If `nums[mid] == target`, return `True`.
2. If `nums[lo] == nums[mid] == nums[hi]`, move `lo` and `hi` inward by one.
3. Otherwise apply the distinct-values logic.

**Complexity**

- **Time:** O(log n) average, O(n) worst (e.g. all values equal).
- **Space:** O(1).

In [44]:
def search_rotated_ii(nums: list[int], target: int) -> bool:
    """True if target is in a rotated sorted array with duplicates."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        # cannot tell which half is sorted: trim both ends
        if nums[lo] == nums[mid] == nums[hi]:
            lo, hi = lo + 1, hi - 1
        # left half sorted
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        # right half sorted
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False


assert search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 0)
assert not search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 3)
assert search_rotated_ii([1, 0, 1, 1, 1], 0)
print(search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 0))

True


## 2.3 Search on the answer

The recipe: (1) say what you are searching for (a speed, a capacity, a day, a value);
(2) give its smallest and largest possible values; (3) write `feasible(v)`, usually a greedy
O(n) scan; (4) check that feasibility is monotone; (5) run the first-true template.

### 2.3.1 [875. Koko Eating Bananas](https://leetcode.com/problems/koko-eating-bananas/) — ★★ · Medium

**Problem.** Minimum eating speed `k` to finish all piles within `h` hours (one pile per hour max).  
**Signal.** "Minimum speed such that feasible" + monotone feasibility → binary search on `k`.  
*New here: Search on the answer with a feasibility function.*

**Fine print.** Ask before coding:
- Can Koko eat from several piles in one hour? → No, at most one pile per hour.
- Is `h ≥` number of piles? → Yes, so an answer always exists.

> 💡 **Key insight.** Eating faster never takes more hours, so "finishes in ≤ h hours" is false for slow speeds and true for fast ones.

**Brute force →** Try speeds 1, 2, 3, … → O(n · max). Binary search over speeds → O(n log max).

**Steps**
1. Speeds range over `[1, max(piles)]`.
2. Hours at speed `k` = `sum(ceil(p / k))`.
3. First speed with hours ≤ `h` via the template.

**Complexity**

- **Time:** O(n log max) — log max candidates, O(n) check each.
- **Space:** O(1).

In [45]:
def min_eating_speed(piles: list[int], h: int) -> int:
    """Slowest speed that finishes within h hours."""
    # speed max(piles) always finishes in len(piles) <= h hours
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        # hours needed at speed mid (ceil per pile)
        if sum((p + mid - 1) // mid for p in piles) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert min_eating_speed([3, 6, 7, 11], 8) == 4
assert min_eating_speed([30, 11, 23, 4, 20], 5) == 30
assert min_eating_speed([30, 11, 23, 4, 20], 6) == 23
print(min_eating_speed([3, 6, 7, 11], 8))

4


### 2.3.2 [1011. Capacity To Ship Packages Within D Days](https://leetcode.com/problems/capacity-to-ship-packages-within-d-days/) — ★★ · Medium

**Problem.** Minimum ship capacity to ship packages in order within `days` days.  
**Signal.** Minimum capacity such that feasible → binary search on capacity, greedy check.  
*New here: A greedy simulation as the feasibility check, and a range with a non-trivial lower end.*

**Fine print.** Ask before coding:
- Must packages be shipped in the given order? → Yes.
- Can a package be split? → No, so capacity is at least `max(weights)`.

> 💡 **Key insight.** For a fixed capacity, loading each day as full as possible uses the fewest days, and a larger capacity never needs more days.

**Brute force →** Try every capacity from `max` to `sum` → O(n · sum). Binary search → O(n log sum).

**Steps**
1. Range `[max(weights), sum(weights)]` (must fit the heaviest package; one day ships all).
2. `days_needed(cap)`: load greedily, start a new day on overflow.
3. First capacity with `days_needed ≤ days`.

**Complexity**

- **Time:** O(n log sum).
- **Space:** O(1).

⚠️ **Trap.** Starting the range at 1 makes `days_needed` wrong for packages heavier than the capacity.

In [46]:
def ship_within_days(weights: list[int], days: int) -> int:
    """Least capacity shipping everything within days."""

    def days_needed(cap: int) -> int:
        count, load = 1, 0
        for w in weights:
            # this package overflows today: start a new day
            if load + w > cap:
                count += 1
                load = 0
            load += w
        return count

    # must carry the heaviest package; one day carries everything
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if days_needed(mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15
assert ship_within_days([3, 2, 2, 4, 1, 4], 3) == 6
assert ship_within_days([1, 2, 3, 1, 1], 4) == 3
print(ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5))

15


### 2.3.3 [410. Split Array Largest Sum](https://leetcode.com/problems/split-array-largest-sum/) — ★ · Hard

**Problem.** Split into `k` non-empty contiguous parts minimising the largest part sum.  
**Signal.** Identical to Ship Within Days: "largest part ≤ cap" is feasible with ≤ k parts.  
*New here: Recognising the same problem in different words.*

**Fine print.** Ask before coding:
- Non-negative values? → Yes, which is what makes greedy feasibility work.
- Non-empty, contiguous parts? → Yes.

> 💡 **Key insight.** "Minimise the largest part" = "smallest cap such that the array splits into ≤ k parts of sum ≤ cap" — exactly the shipping problem.

**Brute force →** Try every split → exponential; interval DP → O(k · n²). Binary search → O(n log sum).

**Steps**
1. Treat parts as days and `nums` as package weights.
2. Reuse the shipping search.

**Complexity**

- **Time:** O(n log sum).
- **Space:** O(1).

In [47]:
def split_array(nums: list[int], k: int) -> int:
    """Minimised largest subarray sum over k parts."""
    # parts = days, part sum limit = ship capacity
    return ship_within_days(nums, k)


assert split_array([7, 2, 5, 10, 8], 2) == 18
assert split_array([1, 2, 3, 4, 5], 2) == 9
print(split_array([7, 2, 5, 10, 8], 2))

18


### 2.3.4 [1482. Minimum Number of Days to Make m Bouquets](https://leetcode.com/problems/minimum-number-of-days-to-make-m-bouquets/) — ★ · Medium

**Problem.** Flower `i` blooms on day `bloom[i]`; a bouquet needs `k` *adjacent* bloomed flowers. Earliest day to make `m` bouquets?  
**Signal.** More days → more bloomed flowers → monotone → binary search on the day.  
*New here: Searching over *time*, plus an early impossibility check.*

**Fine print.** Ask before coding:
- Must flowers be adjacent in a bouquet? → Yes.
- Impossible (`m·k > n`)? → Return −1.

> 💡 **Key insight.** Waiting longer never un-blooms a flower, so "can make m bouquets by day d" is monotone in `d`.

**Brute force →** Simulate every day → O(n · max). Binary search over days → O(n log max).

**Steps**
1. If `m · k > n`, return −1.
2. `bouquets(d)`: count runs of bloomed flowers, one bouquet per `k` in a row.
3. First day with `bouquets(d) ≥ m`.

**Complexity**

- **Time:** O(n log max).
- **Space:** O(1).

In [48]:
def min_days(bloom_day: list[int], m: int, k: int) -> int:
    """Earliest day to form m bouquets of k adjacent flowers."""
    # not enough flowers at all
    if m * k > len(bloom_day):
        return -1

    def bouquets(day: int) -> int:
        count = run = 0
        for b in bloom_day:
            # extend or break the run of bloomed neighbours
            run = run + 1 if b <= day else 0
            if run == k:
                count += 1
                run = 0
        return count

    lo, hi = min(bloom_day), max(bloom_day)
    while lo < hi:
        mid = (lo + hi) // 2
        if bouquets(mid) >= m:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert min_days([1, 10, 3, 10, 2], 3, 1) == 3
assert min_days([1, 10, 3, 10, 2], 3, 2) == -1
assert min_days([7, 7, 7, 7, 12, 7, 7], 2, 3) == 12
print(min_days([1, 10, 3, 10, 2], 3, 1))

3


### 2.3.5 [1283. Find the Smallest Divisor Given a Threshold](https://leetcode.com/problems/find-the-smallest-divisor-given-a-threshold/) — ★ · Medium

**Problem.** Smallest divisor `d` such that `sum(ceil(x / d)) ≤ threshold`.  
**Signal.** Same shape as Koko: monotone cost in `d`.  
*New here: Nothing new — a speed drill for the search-on-answer template.*

**Fine print.** Ask before coding:
- Is ceiling division used? → Yes.
- Is an answer always guaranteed? → Yes, `threshold ≥ n`.

> 💡 **Key insight.** A larger divisor never increases the rounded-up sum, so the condition is monotone in `d`.

**Brute force →** Try every `d` → O(n · max). Binary search → O(n log max).

**Steps**
1. Range `[1, max(nums)]`.
2. Cost(d) = `sum(ceil(x / d))`.
3. First `d` with cost ≤ threshold.

**Complexity**

- **Time:** O(n log max).
- **Space:** O(1).

In [49]:
def smallest_divisor(nums: list[int], threshold: int) -> int:
    """Smallest divisor keeping the rounded-up sum within threshold."""
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        # rounded-up sum shrinks as the divisor grows
        if sum((x + mid - 1) // mid for x in nums) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert smallest_divisor([1, 2, 5, 9], 6) == 5
assert smallest_divisor([44, 22, 33, 11, 1], 5) == 44
print(smallest_divisor([1, 2, 5, 9], 6))

5


### 2.3.6 [378. Kth Smallest Element in a Sorted Matrix](https://leetcode.com/problems/kth-smallest-element-in-a-sorted-matrix/) — ★ · Medium

**Problem.** k-th smallest value in an `n × n` matrix whose rows and columns are sorted.  
**Signal.** Binary search on the *value*, counting elements ≤ value with a staircase walk.  
*New here: Searching on values with a *counting* check.*

**Fine print.** Ask before coding:
- Are duplicates counted? → Yes, the k-th in sorted order, not the k-th distinct.
- Is the matrix square? → Yes, `n × n`.

> 💡 **Key insight.** The k-th smallest is the smallest value `v` with at least `k` entries ≤ `v`, and that count is monotone in `v`.

**Brute force →** Flatten and sort → O(n² log n). Heap of row heads → O(k log n). Value search + staircase count → O(n log range).

**Steps**
1. Range `[matrix[0][0], matrix[-1][-1]]`.
2. `count_le(v)`: from the bottom-left, move right when ≤ v (adding the column count), up otherwise.
3. First `v` with `count_le(v) ≥ k`.

**Complexity**

- **Time:** O(n log(max − min)).
- **Space:** O(1).

In [50]:
def kth_smallest_matrix(matrix: list[list[int]], k: int) -> int:
    """k-th smallest entry of a sorted matrix."""
    n = len(matrix)

    def count_le(v: int) -> int:
        # staircase from the bottom-left corner
        r, c, count = n - 1, 0, 0
        while r >= 0 and c < n:
            # whole column c up to row r is <= v
            if matrix[r][c] <= v:
                count += r + 1
                c += 1
            else:
                r -= 1
        return count

    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert kth_smallest_matrix([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8) == 13
assert kth_smallest_matrix([[-5]], 1) == -5
print(kth_smallest_matrix([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))

13


### 2.3.7 [4. Median of Two Sorted Arrays](https://leetcode.com/problems/median-of-two-sorted-arrays/) — ★★ · Hard

**Problem.** Median of two sorted arrays in O(log(min(m, n))).  
**Signal.** Binary search a *partition* of the shorter array.  
*New here: Binary search over a split point between two arrays.*

**Fine print.** Ask before coding:
- Can one array be empty? → Yes, but not both.
- Even total length? → Return the mean of the two middle values.
- Must it be O(log(m + n))? → Yes; merging is O(m + n).

> 💡 **Key insight.** The median splits the combined data into a left half and a right half; choosing how many of the left half come from `a` fixes the rest, and the split is correct when every left element ≤ every right element.

**Brute force →** Merge and pick the middle → O(m + n). Partition search → O(log min(m, n)).

**Steps**
1. Make `a` the shorter array; `half = (m + n + 1) // 2`.
2. Take `i` from `a` and `j = half − i` from `b` (±∞ beyond the edges).
3. `a[i−1] > b[j]` → take fewer from `a`; `b[j−1] > a[i]` → take more.
4. When valid: odd total → max of left side; even → average of max-left and min-right.

**Complexity**

- **Time:** O(log min(m, n)).
- **Space:** O(1).

In [51]:
def find_median_sorted_arrays(a: list[int], b: list[int]) -> float:
    """Median of the union of two sorted arrays."""
    # search the shorter array's split point
    if len(a) > len(b):
        a, b = b, a
    m, n = len(a), len(b)
    half = (m + n + 1) // 2
    lo, hi = 0, m
    inf = float("inf")
    while True:
        # i items from a and j from b form the left half
        i = (lo + hi) // 2
        j = half - i
        a_left = a[i - 1] if i > 0 else -inf
        a_right = a[i] if i < m else inf
        b_left = b[j - 1] if j > 0 else -inf
        b_right = b[j] if j < n else inf
        # too many from a
        if a_left > b_right:
            hi = i - 1
        # too few from a
        elif b_left > a_right:
            lo = i + 1
        # valid split: read the median from the boundary values
        else:
            left_max = max(a_left, b_left)
            if (m + n) % 2:
                return float(left_max)
            return (left_max + min(a_right, b_right)) / 2


assert find_median_sorted_arrays([1, 3], [2]) == 2.0
assert find_median_sorted_arrays([1, 2], [3, 4]) == 2.5
assert find_median_sorted_arrays([], [1]) == 1.0
print(find_median_sorted_arrays([1, 2], [3, 4]))

2.5


### 2.3.8 [981. Time Based Key-Value Store](https://leetcode.com/problems/time-based-key-value-store/) — ★★ · Medium

**Problem.** `set(key, value, t)`; `get(key, t)` returns the value with the largest timestamp ≤ `t`.  
**Signal.** Timestamps arrive increasing → per-key sorted list + upper-bound search.  
*New here: Binary search inside a data-structure design (an upper bound with `bisect_right`).*

**Fine print.** Ask before coding:
- Do timestamps for `set` strictly increase? → Yes, so each key's list is already sorted.
- No timestamp ≤ `t`? → Return `""`.

> 💡 **Key insight.** Because timestamps per key arrive in increasing order, each key's history is already sorted, and "largest ≤ t" is one position before the upper bound.

**Brute force →** Scan the key's history on every `get` → O(n). `bisect_right` → O(log n).

**Steps**
1. `set` appends the timestamp and value to the key's two lists.
2. `get` computes `i = bisect_right(times, t) − 1`.
3. Return `values[i]`, or `""` if `i < 0`.

**Complexity**

- **Time:** O(1) set, O(log n) get.
- **Space:** O(total sets).

In [52]:
class TimeMap:
    """Versioned key-value store."""

    def __init__(self) -> None:
        # parallel lists per key; timestamps arrive increasing
        self.times: dict[str, list[int]] = defaultdict(list)
        self.values: dict[str, list[str]] = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        # last entry with time <= timestamp
        i = bisect_right(self.times[key], timestamp) - 1
        return self.values[key][i] if i >= 0 else ""


tm = TimeMap()
tm.set("foo", "bar", 1)
assert tm.get("foo", 1) == "bar" and tm.get("foo", 3) == "bar"
tm.set("foo", "bar2", 4)
assert tm.get("foo", 4) == "bar2" and tm.get("foo", 5) == "bar2"
assert tm.get("foo", 0) == ""
print(tm.get("foo", 3))

bar


---
# 3) Linked List

**In plain English.** A linked list is a chain of nodes where each node only knows the next
one. You cannot jump to position i — you can only walk and rewire `next` pointers. Every
question is about rewiring safely without losing the rest of the chain.

> **Analogy.** A treasure hunt: each clue tells you where the next clue is. If you rewrite
> a clue before reading it, the rest of the hunt is lost — so always read (save `next`)
> before you write.

Three tools solve nearly every question:

| Tool | Use it for |
|---|---|
| **Dummy head** `dummy = ListNode(0, head)` | any operation that might change or delete the head |
| **Slow/fast pointers** | middle node, cycle detection, k-th from the end |
| **In-place reversal** `prev, cur = None, head` | reverse all / part / k-groups, palindrome check, reorder |

Update pointers in this order: **save `next` → rewire → advance**. Draw three boxes on paper
for any rewiring with more than two assignments.

**Recognition signals**

- "reverse", "reorder", "swap pairs", "k-group"
- "middle", "cycle", "where the cycle begins", "n-th from the end", "intersection"
- "merge sorted lists", "sort a list in O(n log n)"
- "remove nodes", "partition", "rotate", "deep copy"
- "O(1) cache operations" → hash map + doubly linked list

**Common bugs**

- Losing the rest of the list by rewiring `cur.next` before saving it.
- Special-casing the head instead of using a dummy node.
- `while fast.next` without checking `fast` first → `AttributeError` on `None`.
- Forgetting to cut a list (`slow.next = None`) when splitting it, creating a cycle.
- Comparing nodes with `==` on values when identity (`is`) is meant.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Array two pointers | arrays allow index jumps; lists only allow walking forward |
| Slow/fast (lists) vs same-direction pointers (arrays) | different *speeds* find middles and cycles; arrays use different *conditions* |
| Floyd cycle detection vs visited set | Floyd uses O(1) space; a set is simpler but O(n) |
| LRU (linked list) vs heap | LRU needs O(1) move-to-front; a heap gives O(log n) priority order |

**Learning path.** Reverse (the core rewiring) → merge two → merge k (heap) → partial
reversal → k-groups and pairs → digit arithmetic → merge sort on lists. Then slow/fast:
middle → cycle → cycle start → cycle in an array → gap pointers → palindrome / reorder /
twin sum (middle + reverse combo) → intersection. Then deletion and rewiring, and finally
designs built on doubly linked lists (LRU, LFU, browser history).

In [53]:
class ListNode:
    """Singly linked list node."""

    def __init__(self, val: int = 0, next: "ListNode | None" = None) -> None:
        self.val = val
        self.next = next


def build(values: list[int]) -> ListNode | None:
    """Linked list from a Python list."""
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: ListNode | None) -> list[int]:
    """Python list from a linked list."""
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


print(to_list(build([1, 2, 3])))

[1, 2, 3]


## 3.1 Reversal and merging

Every reversal is the same three-pointer dance (`prev`, `cur`, `nxt`). Partial and group
reversal add one more pointer: the node *before* the segment, so the reversed piece can be
reconnected.

### 3.1.1 [206. Reverse Linked List](https://leetcode.com/problems/reverse-linked-list/) — ★★★ · Easy

**Problem.** Reverse a singly linked list.  
**Signal.** The fundamental pointer rewiring.  
*New here: The save → rewire → advance loop.*

**Fine print.** Ask before coding:
- Iterative or recursive? → Either; iterative is O(1) space.
- Can the list be empty? → Yes.

> 💡 **Key insight.** Reversing means pointing each node at its predecessor; carrying `prev` while walking gives you that predecessor for free.

**Brute force →** Copy values into an array and rebuild → O(n) extra space. In-place rewiring → O(1).

**Steps**
1. `prev = None`, `cur = head`.
2. Save `nxt = cur.next`.
3. Point `cur.next` at `prev`.
4. Advance `prev = cur`, `cur = nxt`; return `prev` at the end.

**Complexity**

- **Time:** O(n) — each node rewired once.
- **Space:** O(1).

In [54]:
def reverse_list(head: ListNode | None) -> ListNode | None:
    """Head of the reversed list."""
    prev = None
    cur = head
    while cur:
        # save the rest of the list before rewiring
        nxt = cur.next
        # point this node backwards
        cur.next = prev
        # advance both pointers
        prev, cur = cur, nxt
    return prev


assert to_list(reverse_list(build([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert to_list(reverse_list(build([]))) == []
print(to_list(reverse_list(build([1, 2]))))

[2, 1]


### 3.1.2 [21. Merge Two Sorted Lists](https://leetcode.com/problems/merge-two-sorted-lists/) — ★★★ · Easy

**Problem.** Merge two sorted lists into one sorted list.  
**Signal.** Merge step of merge sort + dummy head.  
*New here: The dummy head and a moving `tail` to build an output list.*

**Fine print.** Ask before coding:
- Splice existing nodes, or create new ones? → Splice the existing nodes.
- Can either list be empty? → Yes.

> 💡 **Key insight.** The smallest remaining node is always one of the two heads, so attach it and advance that list.

**Brute force →** Copy all values, sort, rebuild → O((m + n) log(m + n)). The merge is linear.

**Steps**
1. `dummy` and `tail` start at a placeholder node.
2. While both lists have nodes, attach the smaller head to `tail` and advance it.
3. Attach whatever list remains.
4. Return `dummy.next`.

**Complexity**

- **Time:** O(m + n).
- **Space:** O(1) — nodes are reused.

In [55]:
def merge_two_lists(a: ListNode | None, b: ListNode | None) -> ListNode | None:
    """Merged sorted list."""
    # placeholder so the first attach needs no special case
    dummy = tail = ListNode()
    while a and b:
        # take the smaller head
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    # one list is empty; append the other as is
    tail.next = a or b
    return dummy.next


assert to_list(merge_two_lists(build([1, 2, 4]), build([1, 3, 4]))) == [
    1,
    1,
    2,
    3,
    4,
    4,
]
assert to_list(merge_two_lists(build([]), build([0]))) == [0]
print(to_list(merge_two_lists(build([1, 2, 4]), build([1, 3, 4]))))

[1, 1, 2, 3, 4, 4]


### 3.1.3 [23. Merge k Sorted Lists](https://leetcode.com/problems/merge-k-sorted-lists/) — ★★★ · Hard

**Problem.** Merge `k` sorted linked lists.  
**Signal.** Repeatedly need the smallest of k heads → min-heap.  
*New here: A heap to pick the smallest of k candidates.*

**Fine print.** Ask before coding:
- Can `lists` be empty or contain empty lists? → Yes, both.
- How large are `k` and the total `N`? → `k ≤ 10⁴` and `N ≤ 10⁴`, so O(N log k) is the target.

> 💡 **Key insight.** The next output node is the smallest of the k current heads; a heap keeps that minimum available in O(log k).

**Brute force →** Merge lists one by one → O(N · k). Pairwise divide and conquer → O(N log k). Heap → O(N log k), simple to write.

**Steps**
1. Push `(val, index, node)` for each non-empty list head.
2. Pop the smallest and attach it to the output.
3. Push that node's successor if it exists.
4. Repeat until the heap is empty.

**Complexity**

- **Time:** O(N log k) — N pops and pushes on a heap of size ≤ k.
- **Space:** O(k) — the heap.

⚠️ **Trap.** Without the index tie-breaker, equal values make Python compare `ListNode`s and crash.

In [56]:
def merge_k_lists(lists: list[ListNode | None]) -> ListNode | None:
    """Merged sorted list of k sorted lists."""
    # index i breaks ties so nodes are never compared
    heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(heap)
    dummy = tail = ListNode()
    while heap:
        # smallest current head across all lists
        _, i, node = heapq.heappop(heap)
        tail.next = tail = node
        # its successor becomes that list's new head
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next


lists = [build([1, 4, 5]), build([1, 3, 4]), build([2, 6])]
assert to_list(merge_k_lists(lists)) == [1, 1, 2, 3, 4, 4, 5, 6]
assert to_list(merge_k_lists([])) == []
print("ok")

ok


### 3.1.4 [92. Reverse Linked List II](https://leetcode.com/problems/reverse-linked-list-ii/) — ★★ · Medium

**Problem.** Reverse the nodes from position `left` to `right` (1-indexed) in one pass.  
**Signal.** Partial reversal → dummy head + "move next node to the front of the sublist".  
*New here: Head insertion: reversing a segment by repeatedly moving a node to its front.*

**Fine print.** Ask before coding:
- 1-indexed with `left ≤ right`? → Yes.
- Can `left` be 1? → Yes, which is why a dummy head helps.

> 💡 **Key insight.** Keep `pre` fixed before the segment and repeatedly lift the node after `cur` to the front of the segment; after `right − left` lifts, the segment is reversed.

**Brute force →** Copy segment values, reverse, write back → O(n) space. Head insertion is O(1).

**Steps**
1. Walk `pre` to the node before position `left` (start at a dummy).
2. `cur = pre.next` (it will end up as the segment's tail).
3. Repeat `right − left` times: detach `cur.next` and insert it right after `pre`.
4. Return `dummy.next`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [57]:
def reverse_between(head: ListNode | None, left: int, right: int) -> ListNode | None:
    """Reverse positions left..right."""
    dummy = ListNode(0, head)
    # pre: node just before the segment
    pre = dummy
    for _ in range(left - 1):
        pre = pre.next
    # cur stays put and drifts to the segment's end
    cur = pre.next
    for _ in range(right - left):
        # lift the node after cur...
        moved = cur.next
        cur.next = moved.next
        # ...and insert it at the front of the segment
        moved.next = pre.next
        pre.next = moved
    return dummy.next


assert to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)) == [1, 4, 3, 2, 5]
assert to_list(reverse_between(build([5]), 1, 1)) == [5]
print(to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)))

[1, 4, 3, 2, 5]


### 3.1.5 [25. Reverse Nodes in k-Group](https://leetcode.com/problems/reverse-nodes-in-k-group/) — ★★ · Hard

**Problem.** Reverse every consecutive group of `k` nodes; a final short group stays as is.  
**Signal.** Repeated partial reversal with a "group previous" pointer.  
*New here: Reversing a bounded segment and reconnecting both of its ends.*

**Fine print.** Ask before coding:
- Does a final group shorter than `k` stay as is? → Yes.
- Swap nodes, not values? → Yes.
- Is O(1) extra space required? → Ideally.

> 💡 **Key insight.** Reverse each group with its successor as the initial `prev`, so the reversed group's tail is already linked to what follows; only the front link needs fixing.

**Brute force →** Collect nodes into arrays of k and relink → O(n) space. In-place group reversal → O(1).

**Steps**
1. From `group_prev`, walk k nodes to find `kth`; fewer than k → stop.
2. Reverse the group, starting `prev` at `kth.next`.
3. Link `group_prev.next = kth` (new group head).
4. The old group head is now the tail: it becomes the next `group_prev`.

**Complexity**

- **Time:** O(n) — each node visited twice.
- **Space:** O(1).

In [58]:
def reverse_k_group(head: ListNode | None, k: int) -> ListNode | None:
    """Reverse nodes in groups of k."""
    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        # find the k-th node of this group, or stop if the group is short
        kth = group_prev
        for _ in range(k):
            kth = kth.next
            if not kth:
                return dummy.next
        group_next = kth.next
        # reverse the group; its tail ends up pointing at group_next
        prev, cur = group_next, group_prev.next
        while cur is not group_next:
            nxt = cur.next
            cur.next = prev
            prev, cur = cur, nxt
        # reconnect the front and move to the next group
        old_head = group_prev.next
        group_prev.next = kth
        group_prev = old_head


assert to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
print(to_list(reverse_k_group(build([1, 2, 3, 4, 5, 6]), 3)))

[3, 2, 1, 6, 5, 4]


### 3.1.6 [24. Swap Nodes in Pairs](https://leetcode.com/problems/swap-nodes-in-pairs/) — ★★ · Medium

**Problem.** Swap every two adjacent nodes (rewire nodes, not values).  
**Signal.** Reverse k-Group with `k = 2`, small enough to write directly.  
*New here: Writing a small rewiring as one simultaneous assignment.*

**Fine print.** Ask before coding:
- Rewire nodes rather than swap values? → Yes.
- Odd length? → The last node stays where it is.

> 💡 **Key insight.** With `prev` before the pair `a, b`, the new order is `prev → b → a → rest`, three pointer changes.

**Brute force →** Swap values instead of nodes → not allowed. The 3-pointer rewiring per pair is O(1).

**Steps**
1. Start `prev` at a dummy.
2. While two nodes follow `prev`, name them `a`, `b`.
3. Rewire `prev → b`, `a → b.next`, `b → a`.
4. Move `prev` to `a`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [59]:
def swap_pairs(head: ListNode | None) -> ListNode | None:
    """Swap adjacent pairs of nodes."""
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a, b = prev.next, prev.next.next
        # prev -> b -> a -> rest (right side is evaluated before assigning)
        prev.next, a.next, b.next = b, b.next, a
        # a is now the second node of the pair
        prev = a
    return dummy.next


assert to_list(swap_pairs(build([1, 2, 3, 4]))) == [2, 1, 4, 3]
assert to_list(swap_pairs(build([1, 2, 3]))) == [2, 1, 3]
print(to_list(swap_pairs(build([1, 2, 3, 4]))))

[2, 1, 4, 3]


### 3.1.7 [2. Add Two Numbers](https://leetcode.com/problems/add-two-numbers/) — ★★ · Medium

**Problem.** Digits stored in reverse order; return the sum as a list.  
**Signal.** Grade-school addition from the least significant digit, carry propagates.  
*New here: Walking two lists of different lengths in lockstep, with a carry.*

**Fine print.** Ask before coding:
- No leading zeros except the number 0 itself? → Yes.
- Different lengths? → Yes, and keep any final carry.

> 💡 **Key insight.** Reverse order means the lists already start at the ones digit — exactly the order you add by hand.

**Brute force →** Convert both to integers and back → fails in fixed-width languages for long lists. Digit-by-digit is O(max(m, n)).

**Steps**
1. Loop while either list has nodes or `carry` is non-zero.
2. `total = carry + a.val + b.val` (missing digits count as 0).
3. `carry, digit = divmod(total, 10)`; append `digit`.
4. Advance whichever lists remain.

**Complexity**

- **Time:** O(max(m, n)).
- **Space:** O(max(m, n)) — the result.

⚠️ **Trap.** Stopping when both lists end loses a final carry (`5 + 5 = 10`).

In [60]:
def add_two_numbers(a: ListNode | None, b: ListNode | None) -> ListNode | None:
    """Sum of two reversed-digit numbers."""
    dummy = tail = ListNode()
    carry = 0
    # keep going while digits or a carry remain
    while a or b or carry:
        total = carry + (a.val if a else 0) + (b.val if b else 0)
        carry, digit = divmod(total, 10)
        tail.next = tail = ListNode(digit)
        a = a.next if a else None
        b = b.next if b else None
    return dummy.next


assert to_list(add_two_numbers(build([2, 4, 3]), build([5, 6, 4]))) == [7, 0, 8]
assert to_list(add_two_numbers(build([9] * 7), build([9] * 4))) == [
    8,
    9,
    9,
    9,
    0,
    0,
    0,
    1,
]
print(to_list(add_two_numbers(build([2, 4, 3]), build([5, 6, 4]))))

[7, 0, 8]


### 3.1.8 [445. Add Two Numbers II](https://leetcode.com/problems/add-two-numbers-ii/) — ★ · Medium

**Problem.** Same as Add Two Numbers, but digits are stored most-significant first.  
**Signal.** Need digits from the end → stacks (or reverse both lists).  
*New here: Building the result by *prepending*, so it comes out in forward order.*

**Fine print.** Ask before coding:
- Can I reverse the input lists? → The follow-up says no, so use stacks.
- Different lengths? → Yes.

> 💡 **Key insight.** Addition must start from the ones digit, which is at the end; stacks give the digits in that order, and prepending each new digit keeps the output forward.

**Brute force →** Reverse both lists, add, reverse the result → O(m + n), three passes. Stacks + prepending → two passes, no reversal.

**Steps**
1. Put each list's digits into a stack (a Python list).
2. Pop from both, add with carry.
3. Prepend a node with the digit to the result.
4. Continue while digits or a carry remain.

**Complexity**

- **Time:** O(m + n).
- **Space:** O(m + n) — the stacks.

In [61]:
def add_two_numbers_ii(a: ListNode | None, b: ListNode | None) -> ListNode | None:
    """Sum of two forward-digit numbers."""
    # digit stacks: pop() yields the least significant digit first
    s1, s2 = to_list(a), to_list(b)
    head, carry = None, 0
    while s1 or s2 or carry:
        total = carry + (s1.pop() if s1 else 0) + (s2.pop() if s2 else 0)
        carry, digit = divmod(total, 10)
        # prepend so the most significant digit ends up first
        head = ListNode(digit, head)
    return head


assert to_list(add_two_numbers_ii(build([7, 2, 4, 3]), build([5, 6, 4]))) == [
    7,
    8,
    0,
    7,
]
assert to_list(add_two_numbers_ii(build([0]), build([0]))) == [0]
print(to_list(add_two_numbers_ii(build([7, 2, 4, 3]), build([5, 6, 4]))))

[7, 8, 0, 7]


### 3.1.9 [148. Sort List](https://leetcode.com/problems/sort-list/) — ★★ · Medium

**Problem.** Sort a linked list in O(n log n).  
**Signal.** Linked lists favour merge sort (no random access needed).  
*New here: Combining middle-finding, splitting and merging into merge sort.*

**Fine print.** Ask before coding:
- Is O(1) extra space required? → The follow-up asks for it, which means bottom-up merge sort.
- Stable sort? → Not required.

> 💡 **Key insight.** Merge sort only needs "split in half" and "merge two sorted lists", both natural on linked lists.

**Brute force →** Copy to an array, sort, rebuild → O(n) extra space. Quicksort on lists is awkward. Merge sort fits.

**Steps**
1. Base case: 0 or 1 node.
2. Find the middle with slow/fast and cut the list in two.
3. Sort each half recursively.
4. Merge the two sorted halves.

**Complexity**

- **Time:** O(n log n) — log n levels, O(n) merging per level.
- **Space:** O(log n) — recursion depth (bottom-up merging achieves O(1)).

In [62]:
def sort_list(head: ListNode | None) -> ListNode | None:
    """Merge-sorted list."""
    if not head or not head.next:
        return head
    # slow stops at the end of the first half
    slow, fast = head, head.next
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    # cut the list into two halves
    right = slow.next
    slow.next = None
    return merge_two_lists(sort_list(head), sort_list(right))


assert to_list(sort_list(build([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_list(sort_list(build([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]
print(to_list(sort_list(build([4, 2, 1, 3]))))

[1, 2, 3, 4]


## 3.2 Slow and fast pointers

Two pointers moving at different speeds (1 and 2 steps) or with a fixed gap (n steps).
Speed difference finds middles and cycles; a fixed gap finds the n-th node from the end.

### 3.2.1 [876. Middle of the Linked List](https://leetcode.com/problems/middle-of-the-linked-list/) — ★★ · Easy

**Problem.** Return the middle node (second middle if even length).  
**Signal.** Fast moves two steps per slow step → slow ends in the middle.  
*New here: The slow/fast walk itself.*

**Fine print.** Ask before coding:
- Even length? → Return the second middle node.

> 💡 **Key insight.** When the fast pointer has walked the whole list, the slow pointer has walked half of it.

**Brute force →** Count the length, then walk n/2 → two passes. Slow/fast → one pass.

**Steps**
1. Start both at the head.
2. Move slow one step and fast two steps while fast and fast.next exist.
3. Return slow.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [63]:
def middle_node(head: ListNode | None) -> ListNode | None:
    """Middle node (second of two middles)."""
    slow = fast = head
    # fast covers twice the distance of slow
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow


assert to_list(middle_node(build([1, 2, 3, 4, 5]))) == [3, 4, 5]
assert to_list(middle_node(build([1, 2, 3, 4, 5, 6]))) == [4, 5, 6]
print(middle_node(build([1, 2, 3])).val)

2


### 3.2.2 [141. Linked List Cycle](https://leetcode.com/problems/linked-list-cycle/) — ★★★ · Easy

**Problem.** Does the list contain a cycle? O(1) memory.  
**Signal.** Floyd's tortoise and hare.  
*New here: Using speed difference to detect a loop without memory.*

**Fine print.** Ask before coding:
- Is `pos` given to you? → No, it is only used to build the test case.
- Is O(1) memory required? → Yes, so use Floyd's fast and slow pointers.

> 💡 **Key insight.** Inside a loop the fast pointer gains one node per step on the slow one, so it must eventually land on it.

**Brute force →** A set of visited nodes → O(n) space. Floyd → O(1).

**Steps**
1. Slow moves 1, fast moves 2.
2. If they ever point to the same node, there is a cycle.
3. If fast reaches `None`, there is none.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [64]:
def has_cycle(head: ListNode | None) -> bool:
    """True if the list loops."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        # fast lapped slow: there is a loop
        if slow is fast:
            return True
    # fast fell off the end
    return False


def build_cycle(values: list[int], pos: int) -> ListNode | None:
    """List whose tail links back to index pos (-1 for none)."""
    head = build(values)
    if pos >= 0:
        nodes = []
        node = head
        while node:
            nodes.append(node)
            node = node.next
        # close the loop
        nodes[-1].next = nodes[pos]
    return head


assert has_cycle(build_cycle([3, 2, 0, -4], 1))
assert not has_cycle(build_cycle([1], -1))
print(has_cycle(build_cycle([1, 2], 0)))

True


### 3.2.3 [142. Linked List Cycle II](https://leetcode.com/problems/linked-list-cycle-ii/) — ★ · Medium

**Problem.** Return the node where the cycle begins, or `None`.  
**Signal.** Floyd phase 2: after meeting, a pointer from the head and one from the meeting point meet at the cycle start.  
*New here: Floyd's second phase and the distance argument behind it.*

**Fine print.** Ask before coding:
- Can I modify the list? → No.
- Is O(1) memory required? → Yes.

> 💡 **Key insight.** With tail length `a`, meeting point `b` into a cycle of length `c`: `2(a + b) = a + b + kc` gives `a = kc − b`, so walking `a` steps from the meeting point lands on the cycle start.

**Brute force →** The first node seen twice in a visited set → O(n) space. Floyd phase 2 → O(1).

**Steps**
1. Run slow/fast until they meet (or fast ends → no cycle).
2. Start `finder` at the head.
3. Move `finder` and `slow` one step at a time.
4. They meet at the cycle start.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [65]:
def detect_cycle(head: ListNode | None) -> ListNode | None:
    """First node of the cycle, if any."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            # head and meeting point are the same distance from the cycle start
            finder = head
            while finder is not slow:
                finder, slow = finder.next, slow.next
            return finder
    return None


assert detect_cycle(build_cycle([3, 2, 0, -4], 1)).val == 2
assert detect_cycle(build_cycle([1], -1)) is None
print(detect_cycle(build_cycle([1, 2], 0)).val)

1


### 3.2.4 [287. Find the Duplicate Number](https://leetcode.com/problems/find-the-duplicate-number/) — ★★ · Medium

**Problem.** `n + 1` integers in `1..n`, one value repeated; find it with O(1) space, no modification.  
**Signal.** Treat `i → nums[i]` as a linked list; the duplicate is the cycle entrance.  
*New here: Seeing a linked list hidden inside an array.*

**Fine print.** Ask before coding:
- Can I modify the array? → No, so cyclic sort is ruled out.
- Is O(1) extra space required? → Yes.
- Can the duplicate appear more than twice? → Yes.

> 💡 **Key insight.** Following `i → nums[i]` is a walk on a graph where two indices point to the duplicate value, so the walk falls into a cycle whose entrance is the duplicate.

**Brute force →** A set → O(n) space. Sorting → modifies the array. Binary search on value counts → O(n log n). Floyd → O(n) time, O(1) space.

**Steps**
1. Slow `x = nums[x]`, fast `x = nums[nums[x]]`, starting from index 0, until they meet.
2. Start `finder` at 0.
3. Advance `finder` and `slow` one step at a time until they meet.
4. That value is the duplicate.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [66]:
def find_duplicate(nums: list[int]) -> int:
    """The repeated value."""
    # phase 1: meet inside the cycle of i -> nums[i]
    slow = fast = 0
    while True:
        slow, fast = nums[slow], nums[nums[fast]]
        if slow == fast:
            break
    # phase 2: walk to the cycle entrance (the duplicate)
    finder = 0
    while finder != slow:
        finder, slow = nums[finder], nums[slow]
    return finder


assert find_duplicate([1, 3, 4, 2, 2]) == 2
assert find_duplicate([3, 1, 3, 4, 2]) == 3
assert find_duplicate([3, 3, 3, 3, 3]) == 3
print(find_duplicate([1, 3, 4, 2, 2]))

2


### 3.2.5 [19. Remove Nth Node From End of List](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) — ★★★ · Medium

**Problem.** Remove the n-th node from the end in one pass.  
**Signal.** Gap of n between two pointers.  
*New here: A fixed gap instead of a speed difference.*

**Fine print.** Ask before coding:
- Is `n` always valid (1 ≤ n ≤ length)? → Yes.
- One pass? → That is the follow-up.

> 💡 **Key insight.** If `fast` is n nodes ahead of `slow`, then when `fast` is on the last node, `slow` is just before the n-th node from the end.

**Brute force →** Count the length, then walk `L − n` → two passes. The gap → one pass.

**Steps**
1. Start both at a dummy before the head.
2. Move `fast` n steps ahead.
3. Move both until `fast.next` is `None`.
4. Unlink `slow.next`.

**Complexity**

- **Time:** O(L).
- **Space:** O(1).

⚠️ **Trap.** Without the dummy, removing the head (n = length) needs a special case.

In [67]:
def remove_nth_from_end(head: ListNode | None, n: int) -> ListNode | None:
    """List without its n-th node from the end."""
    # dummy makes deleting the head a normal case
    dummy = ListNode(0, head)
    slow = fast = dummy
    # open a gap of n nodes
    for _ in range(n):
        fast = fast.next
    # slide until fast is the last node
    while fast.next:
        slow, fast = slow.next, fast.next
    # slow is just before the target
    slow.next = slow.next.next
    return dummy.next


assert to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_list(remove_nth_from_end(build([1]), 1)) == []
assert to_list(remove_nth_from_end(build([1, 2]), 2)) == [2]
print(to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)))

[1, 2, 3, 5]


### 3.2.6 [234. Palindrome Linked List](https://leetcode.com/problems/palindrome-linked-list/) — ★★ · Easy

**Problem.** Is the list a palindrome? O(1) space.  
**Signal.** Middle + reverse second half + compare.  
*New here: The "middle + reverse" combo used by the next questions too.*

**Fine print.** Ask before coding:
- Is O(1) space required? → The follow-up asks for it, so reverse the second half.
- Should the list be restored afterwards? → It is good practice to mention it.

> 💡 **Key insight.** Reversing the second half makes it readable from the end, so the two halves can be compared front to front.

**Brute force →** Copy values to an array and compare with its reverse → O(n) space. Middle + reverse → O(1).

**Steps**
1. Find the middle with slow/fast.
2. Reverse from the middle on.
3. Walk the first half and the reversed half together, comparing values.

**Complexity**

- **Time:** O(n).
- **Space:** O(1) — the list is modified; restore it if the caller needs it.

In [68]:
def is_palindrome_list(head: ListNode | None) -> bool:
    """True if values read the same both ways."""
    # slow lands on the start of the second half
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    # second half reversed, read from the back
    left, right = head, reverse_list(slow)
    while right:
        if left.val != right.val:
            return False
        left, right = left.next, right.next
    return True


assert is_palindrome_list(build([1, 2, 2, 1]))
assert not is_palindrome_list(build([1, 2]))
assert is_palindrome_list(build([1, 2, 3, 2, 1]))
print(is_palindrome_list(build([1, 2, 2, 1])))

True


### 3.2.7 [143. Reorder List](https://leetcode.com/problems/reorder-list/) — ★★★ · Medium

**Problem.** Reorder `L0 → L1 → … → Ln` into `L0 → Ln → L1 → Ln−1 → …` in place.  
**Signal.** Middle + reverse second half + interleave merge.  
*New here: Three building blocks chained: middle, reverse, interleave.*

**Fine print.** Ask before coding:
- Rewire nodes, not values? → Yes.
- Return anything? → No, change the list in place.

> 💡 **Key insight.** The target order alternates the front half with the back half read backwards, so reverse the back half and zip the two halves.

**Brute force →** Copy nodes into an array and relink from both ends → O(n) space. The three-step method → O(1).

**Steps**
1. Find the middle; cut the list after it.
2. Reverse the second half.
3. Alternately take one node from each half until the second is used up.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [69]:
def reorder_list(head: ListNode | None) -> None:
    """Interleave front and reversed back halves in place."""
    # slow ends at the last node of the first half
    slow, fast = head, head.next
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    # detach and reverse the second half
    second = reverse_list(slow.next)
    slow.next = None
    # zip: one node from each half in turn
    first = head
    while second:
        f_next, s_next = first.next, second.next
        first.next, second.next = second, f_next
        first, second = f_next, s_next


h = build([1, 2, 3, 4, 5])
reorder_list(h)
assert to_list(h) == [1, 5, 2, 4, 3]
h = build([1, 2, 3, 4])
reorder_list(h)
assert to_list(h) == [1, 4, 2, 3]
print(to_list(h))

[1, 4, 2, 3]


### 3.2.8 [160. Intersection of Two Linked Lists](https://leetcode.com/problems/intersection-of-two-linked-lists/) — ★★ · Easy

**Problem.** Node where two lists merge, or `None`; O(1) space.  
**Signal.** Equalise path lengths by switching heads.  
*New here: Equalising two path lengths by walking both lists.*

**Fine print.** Ask before coding:
- Do nodes or values define the intersection? → Nodes, by identity.
- Can the lists have cycles? → No.
- Must the original structure be kept? → Yes.

> 💡 **Key insight.** Walking A then B and B then A, both pointers travel `len(A) + len(B)` steps, so they reach the shared part at the same moment.

**Brute force →** Store A's nodes in a set and walk B → O(m) space. Compute both lengths and align → O(1) space but more code. Head switching is shortest.

**Steps**
1. Pointer `a` walks A, then continues on B.
2. Pointer `b` walks B, then continues on A.
3. Stop when they are the same node (the intersection, or both `None`).

**Complexity**

- **Time:** O(m + n).
- **Space:** O(1).

In [70]:
def get_intersection_node(head_a: ListNode | None, head_b: ListNode | None):
    """Shared node of two lists, or None."""
    a, b = head_a, head_b
    # both walk len(A) + len(B) steps, so they align on the shared tail
    while a is not b:
        a = a.next if a else head_b
        b = b.next if b else head_a
    return a


shared = build([8, 4, 5])
la = ListNode(4, ListNode(1, shared))
lb = ListNode(5, ListNode(6, ListNode(1, shared)))
assert get_intersection_node(la, lb) is shared
assert get_intersection_node(build([1]), build([2])) is None
print(get_intersection_node(la, lb).val)

8


### 3.2.9 [2130. Maximum Twin Sum of a Linked List](https://leetcode.com/problems/maximum-twin-sum-of-a-linked-list/) — ★ · Medium

**Problem.** Even length; twin of node `i` is node `n−1−i`. Max twin sum?  
**Signal.** Middle + reverse second half + walk both halves.  
*New here: The middle + reverse combo, reused for a different aggregate.*

**Fine print.** Ask before coding:
- Is the length always even? → Yes.
- O(1) space? → Possible by reversing the second half.

> 💡 **Key insight.** After reversing the second half, the i-th node of each half are twins.

**Brute force →** Copy to an array and pair `i` with `n−1−i` → O(n) space. Middle + reverse → O(1).

**Steps**
1. Find the middle.
2. Reverse the second half.
3. Walk both halves together, tracking the maximum pair sum.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [71]:
def pair_sum(head: ListNode | None) -> int:
    """Largest sum of twin nodes."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    # reversed second half lines twins up
    left, right, best = head, reverse_list(slow), 0
    while right:
        best = max(best, left.val + right.val)
        left, right = left.next, right.next
    return best


assert pair_sum(build([5, 4, 2, 1])) == 6
assert pair_sum(build([4, 2, 2, 3])) == 7
print(pair_sum(build([1, 100000])))

100001


## 3.3 Rewiring and deletion

Deleting or moving nodes is "skip over" (`prev.next = prev.next.next`) or "build new chains
with dummy heads and join them". Use a dummy whenever the head might be removed, and always
terminate the last chain with `None`.

### 3.3.1 [83. Remove Duplicates from Sorted List](https://leetcode.com/problems/remove-duplicates-from-sorted-list/) — ★ · Easy

**Problem.** Delete duplicates so each value appears once.  
**Signal.** Sorted → duplicates are adjacent.  
*New here: Deletion by skipping a node.*

**Fine print.** Ask before coding:
- Is the list sorted? → Yes, so duplicates are adjacent.

> 💡 **Key insight.** In a sorted list a duplicate is always the very next node, so skipping it is one pointer change.

**Brute force →** Copy unique values into a new list → O(n) space. Skipping in place → O(1).

**Steps**
1. Walk with `cur`.
2. If `cur.next` has the same value, unlink it (stay on `cur`).
3. Otherwise advance `cur`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [72]:
def delete_duplicates(head: ListNode | None) -> ListNode | None:
    """Sorted list with each value once."""
    cur = head
    while cur and cur.next:
        # duplicate neighbour: skip it but stay to check the new neighbour
        if cur.next.val == cur.val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return head


assert to_list(delete_duplicates(build([1, 1, 2, 3, 3]))) == [1, 2, 3]
print(to_list(delete_duplicates(build([1, 1, 2]))))

[1, 2]


### 3.3.2 [82. Remove Duplicates from Sorted List II](https://leetcode.com/problems/remove-duplicates-from-sorted-list-ii/) — ★★ · Medium

**Problem.** Delete *all* nodes that have duplicate values, keeping only distinct values.  
**Signal.** Head may be deleted → dummy; skip whole runs.  
*New here: Skipping a whole run, with `prev` as the last *kept* node.*

**Fine print.** Ask before coding:
- Remove every copy of a repeated value? → Yes, keep only values that appear once.
- Can the head be removed? → Yes, so use a dummy node.

> 💡 **Key insight.** Look ahead from the last kept node: if the next two values are equal, the whole run must go; otherwise the next node is safe to keep.

**Brute force →** Count values, then rebuild keeping count-1 values → O(n) space. One pass with a dummy → O(1).

**Steps**
1. `prev` starts at a dummy before the head.
2. If `prev.next` starts a run of equal values, walk to the run's end and link `prev` past it.
3. Otherwise advance `prev`.
4. Return `dummy.next`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [73]:
def delete_duplicates_ii(head: ListNode | None) -> ListNode | None:
    """Sorted list keeping only values that appear once."""
    dummy = ListNode(0, head)
    # prev = last node known to be kept
    prev = dummy
    while prev.next:
        cur = prev.next
        # cur starts a run of equal values: drop the whole run
        if cur.next and cur.next.val == cur.val:
            while cur.next and cur.next.val == cur.val:
                cur = cur.next
            prev.next = cur.next
        # cur is unique: keep it
        else:
            prev = cur
    return dummy.next


assert to_list(delete_duplicates_ii(build([1, 2, 3, 3, 4, 4, 5]))) == [1, 2, 5]
assert to_list(delete_duplicates_ii(build([1, 1, 1, 2, 3]))) == [2, 3]
print(to_list(delete_duplicates_ii(build([1, 2, 3, 3, 4, 4, 5]))))

[1, 2, 5]


### 3.3.3 [203. Remove Linked List Elements](https://leetcode.com/problems/remove-linked-list-elements/) — ★ · Easy

**Problem.** Remove all nodes with value `val`.  
**Signal.** Head may be removed → dummy head.  
*New here: The dummy head removes the special case of deleting the head.*

**Fine print.** Ask before coding:
- Can the head nodes have value `val`? → Yes, so use a dummy node.

> 💡 **Key insight.** With a dummy in front, every node — including the head — has a predecessor, so deletion is always `cur.next = cur.next.next`.

**Brute force →** Handle the head in a separate loop first → more code, same cost. The dummy unifies the cases.

**Steps**
1. `cur` starts at a dummy before the head.
2. If `cur.next.val == val`, skip it.
3. Otherwise advance.
4. Return `dummy.next`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [74]:
def remove_elements(head: ListNode | None, val: int) -> ListNode | None:
    """List without nodes equal to val."""
    dummy = cur = ListNode(0, head)
    while cur.next:
        # unlink matching nodes; stay to check the new next
        if cur.next.val == val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return dummy.next


assert to_list(remove_elements(build([1, 2, 6, 3, 4, 5, 6]), 6)) == [1, 2, 3, 4, 5]
assert to_list(remove_elements(build([7, 7, 7]), 7)) == []
print(to_list(remove_elements(build([1, 2, 6, 3]), 6)))

[1, 2, 3]


### 3.3.4 [61. Rotate List](https://leetcode.com/problems/rotate-list/) — ★★ · Medium

**Problem.** Rotate the list to the right by `k` places.  
**Signal.** Rotation = close into a ring, then cut at the right place.  
*New here: Turning a list into a ring and cutting it.*

**Fine print.** Ask before coding:
- Can `k` be larger than the length? → Yes, up to 2·10⁹, so take `k % length`.
- Empty list? → Return it as is.

> 💡 **Key insight.** Rotating right by `k` means the new head is the `(n − k % n)`-th node; closing the list into a ring makes that a single cut.

**Brute force →** Move the tail to the front k times → O(n · k). Ring + cut → O(n).

**Steps**
1. Find the length `n` and the tail.
2. Link the tail to the head (ring).
3. Walk `n − k % n − 1` steps from the head to the new tail.
4. The node after it is the new head; cut the ring.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [75]:
def rotate_right(head: ListNode | None, k: int) -> ListNode | None:
    """List rotated right by k."""
    if not head:
        return head
    # length and tail
    n, tail = 1, head
    while tail.next:
        tail, n = tail.next, n + 1
    # close the ring
    tail.next = head
    # new tail sits n - k % n - 1 steps from the old head
    new_tail = head
    for _ in range(n - k % n - 1):
        new_tail = new_tail.next
    # cut the ring after the new tail
    new_head = new_tail.next
    new_tail.next = None
    return new_head


assert to_list(rotate_right(build([1, 2, 3, 4, 5]), 2)) == [4, 5, 1, 2, 3]
assert to_list(rotate_right(build([0, 1, 2]), 4)) == [2, 0, 1]
print(to_list(rotate_right(build([1, 2, 3, 4, 5]), 2)))

[4, 5, 1, 2, 3]


### 3.3.5 [86. Partition List](https://leetcode.com/problems/partition-list/) — ★★ · Medium

**Problem.** Move nodes `< x` before nodes `≥ x`, preserving relative order.  
**Signal.** Stable partition → build two lists with two dummies, then join.  
*New here: Building two chains with two dummies and joining them.*

**Fine print.** Ask before coding:
- Must relative order be kept in both parts? → Yes.
- Do values equal to `x` go to the second part? → Yes.

> 💡 **Key insight.** Appending each node to the end of one of two chains keeps the original order inside each chain automatically.

**Brute force →** Copy values to two arrays and rebuild → O(n) space. Two chains of the original nodes → O(1).

**Steps**
1. Two dummies: `before` and `after`.
2. Append each node to `before` if `< x`, else to `after`.
3. End `after` with `None`.
4. Link `before`'s tail to `after`'s first node.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

⚠️ **Trap.** Forgetting `a_tail.next = None` leaves an old pointer that can form a cycle.

In [76]:
def partition(head: ListNode | None, x: int) -> ListNode | None:
    """Stable partition around x."""
    before = b_tail = ListNode()
    after = a_tail = ListNode()
    while head:
        # append the node to the chain it belongs to
        if head.val < x:
            b_tail.next = b_tail = head
        else:
            a_tail.next = a_tail = head
        head = head.next
    # terminate the second chain, then join the two
    a_tail.next = None
    b_tail.next = after.next
    return before.next


assert to_list(partition(build([1, 4, 3, 2, 5, 2]), 3)) == [1, 2, 2, 4, 3, 5]
assert to_list(partition(build([2, 1]), 2)) == [1, 2]
print(to_list(partition(build([1, 4, 3, 2, 5, 2]), 3)))

[1, 2, 2, 4, 3, 5]


### 3.3.6 [328. Odd Even Linked List](https://leetcode.com/problems/odd-even-linked-list/) — ★★ · Medium

**Problem.** Group all odd-positioned nodes, then even-positioned nodes, in place.  
**Signal.** Two interleaved chains, joined at the end.  
*New here: Splitting one list into two interleaved chains in place.*

**Fine print.** Ask before coding:
- Odd and even by position, not by value? → By position (1-indexed).
- Is O(1) extra space required? → Yes.

> 💡 **Key insight.** Each odd node's new `next` is the node after its even neighbour, and vice versa, so two tails can leapfrog each other.

**Brute force →** Two new lists by position → same idea with dummies. Leapfrogging uses the original head as the odd chain's head.

**Steps**
1. `odd = head`, `even = head.next`, remember `even_head`.
2. While `even` and `even.next` exist: `odd.next = even.next`, advance `odd`.
3. `even.next = odd.next`, advance `even`.
4. Attach `even_head` after the last odd node.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [77]:
def odd_even_list(head: ListNode | None) -> ListNode | None:
    """Odd positions followed by even positions."""
    if not head:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        # odd tail jumps over the even node
        odd.next = even.next
        odd = odd.next
        # even tail jumps over the new odd node
        even.next = odd.next
        even = even.next
    # evens follow the odds
    odd.next = even_head
    return head


assert to_list(odd_even_list(build([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4]
assert to_list(odd_even_list(build([2, 1, 3, 5, 6, 4, 7]))) == [2, 3, 6, 7, 1, 5, 4]
print(to_list(odd_even_list(build([1, 2, 3, 4, 5]))))

[1, 3, 5, 2, 4]


### 3.3.7 [138. Copy List with Random Pointer](https://leetcode.com/problems/copy-list-with-random-pointer/) — ★★ · Medium

**Problem.** Deep-copy a list where each node also has a `random` pointer.  
**Signal.** Map old node → new node, then wire pointers.  
*New here: Copying a graph-like structure with an old → new map.*

**Fine print.** Ask before coding:
- Can `random` be `None` or point to itself? → Yes, both.
- Is O(1) extra space (besides the copy) required? → The interleaving trick achieves it; a hashmap is O(n).

> 💡 **Key insight.** A pointer can only be copied once its target's copy exists, so create all copies first, then wire `next` and `random` through the map.

**Brute force →** For each node, search the copy list to find its random target's position → O(n²). The map makes it O(n).

**Steps**
1. First pass: map every original node to a new node (`None → None` too).
2. Second pass: `copy.next = map[old.next]`, `copy.random = map[old.random]`.
3. Return `map[head]`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) — the map (an interleaving trick gets O(1)).

In [78]:
class RandomNode:
    """Node with an extra random pointer."""

    def __init__(self, val: int) -> None:
        self.val = val
        self.next: "RandomNode | None" = None
        self.random: "RandomNode | None" = None


def copy_random_list(head: RandomNode | None) -> RandomNode | None:
    """Deep copy preserving random links."""
    # old node -> its copy; None maps to None so links can be copied blindly
    copies: dict = {None: None}
    node = head
    while node:
        copies[node] = RandomNode(node.val)
        node = node.next
    # every copy exists now: wire both pointers through the map
    node = head
    while node:
        copies[node].next = copies[node.next]
        copies[node].random = copies[node.random]
        node = node.next
    return copies[head]


nodes = [RandomNode(v) for v in [7, 13, 11, 10, 1]]
for a, b in zip(nodes, nodes[1:]):
    a.next = b
for node, r in zip(nodes, [None, 0, 4, 2, 0]):
    node.random = nodes[r] if r is not None else None
copy = copy_random_list(nodes[0])
out = []
while copy:
    out.append((copy.val, copy.random.val if copy.random else None))
    assert copy not in nodes
    copy = copy.next
assert out == [(7, None), (13, 7), (11, 1), (10, 11), (1, 7)]
print(out)

[(7, None), (13, 7), (11, 1), (10, 11), (1, 7)]


## 3.4 Linked-list design

Design questions combine a hash map (O(1) *find*) with a linked list (O(1) *reorder /
remove* once you have the node). Sentinel head and tail nodes remove every edge case.

### 3.4.1 [146. LRU Cache](https://leetcode.com/problems/lru-cache/) — ★★ · Medium

**Problem.** Fixed-capacity cache with O(1) `get` and `put`, evicting the least recently used key.  
**Signal.** O(1) lookup + O(1) recency order → hash map + doubly linked list.  
*New here: Hash map + doubly linked list with sentinels.*

**Fine print.** Ask before coding:
- Does `put` on an existing key update it and make it most recent? → Yes.
- Does `get` count as a use? → Yes.
- Is capacity ≥ 1? → Yes.

> 💡 **Key insight.** The map finds a node in O(1), and a doubly linked node can be unlinked and moved to the "most recent" end in O(1) because it knows both neighbours.

**Brute force →** A list of keys ordered by use → O(n) to move a key. A timestamp per key + scan for the oldest → O(n) eviction. Map + doubly linked list → O(1) everything.

**Steps**
1. Map `key → node`; list ordered least → most recent between `head` and `tail` sentinels.
2. `get`: if present, move the node to the end and return its value.
3. `put`: remove an existing node, append the new node at the end.
4. Over capacity → remove the node after `head` and delete its key.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(capacity).

⚠️ **Trap.** The node must store its key, or you cannot delete the evicted entry from the map. (`OrderedDict.move_to_end` is the library shortcut.)

In [79]:
class DNode:
    """Doubly linked node for the cache."""

    def __init__(self, key: int = 0, val: int = 0) -> None:
        self.key, self.val = key, val
        self.prev: "DNode | None" = None
        self.next: "DNode | None" = None


class LRUCache:
    """Least-recently-used cache."""

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.map: dict[int, DNode] = {}
        # sentinels: head.next = least recent, tail.prev = most recent
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node: DNode) -> None:
        # neighbours skip over node
        node.prev.next, node.next.prev = node.next, node.prev

    def _append(self, node: DNode) -> None:
        # insert just before the tail sentinel
        node.prev, node.next = self.tail.prev, self.tail
        self.tail.prev.next = node
        self.tail.prev = node

    def get(self, key: int) -> int:
        if key not in self.map:
            return -1
        # mark as most recently used
        node = self.map[key]
        self._remove(node)
        self._append(node)
        return node.val

    def put(self, key: int, value: int) -> None:
        if key in self.map:
            self._remove(self.map[key])
        node = DNode(key, value)
        self.map[key] = node
        self._append(node)
        # evict the least recently used entry
        if len(self.map) > self.capacity:
            lru = self.head.next
            self._remove(lru)
            del self.map[lru.key]


cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
assert cache.get(1) == 1
cache.put(3, 3)
assert cache.get(2) == -1
cache.put(4, 4)
assert [cache.get(1), cache.get(3), cache.get(4)] == [-1, 3, 4]
print("ok")

ok


### 3.4.2 [460. LFU Cache](https://leetcode.com/problems/lfu-cache/) — ★ · Hard

**Problem.** Cache evicting the least *frequently* used key (ties → least recently used), O(1) ops.  
**Signal.** Frequency buckets, each an ordered set of keys, plus a `min_freq` pointer.  
*New here: Buckets of LRU lists, one per frequency, plus a `min_freq` pointer.*

**Fine print.** Ask before coding:
- How are frequency ties broken? → Least recently used.
- Does `put` on an existing key increase its frequency? → Yes.
- Can capacity be 0? → Yes, then `put` does nothing.

> 💡 **Key insight.** Frequencies only grow by one at a time, so the minimum frequency can be tracked in O(1): it only changes when its bucket empties (+1) or a new key arrives (reset to 1).

**Brute force →** A heap keyed by `(freq, time)` → O(log n) per op. Frequency buckets with ordered keys → O(1).

**Steps**
1. `entries: key → [value, freq]`; `buckets: freq → OrderedDict` (insertion order = recency).
2. Touching a key moves it from bucket `f` to `f + 1`; if bucket `min_freq` empties, `min_freq += 1`.
3. New key at capacity: evict the oldest key in bucket `min_freq`.
4. Insert the new key in bucket 1 and set `min_freq = 1`.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(capacity).

In [80]:
class LFUCache:
    """Least-frequently-used cache with LRU tie-break."""

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        # key -> [value, frequency]
        self.entries: dict[int, list[int]] = {}
        # frequency -> keys in least-to-most recent order
        self.buckets: dict[int, OrderedDict] = defaultdict(OrderedDict)
        self.min_freq = 0

    def _touch(self, key: int) -> None:
        # move key from bucket f to bucket f + 1
        value, freq = self.entries[key]
        del self.buckets[freq][key]
        if not self.buckets[freq] and self.min_freq == freq:
            self.min_freq += 1
        self.entries[key] = [value, freq + 1]
        self.buckets[freq + 1][key] = None

    def get(self, key: int) -> int:
        if key not in self.entries:
            return -1
        self._touch(key)
        return self.entries[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        if key in self.entries:
            self.entries[key][0] = value
            self._touch(key)
            return
        # full: evict the least recent key among the least frequent
        if len(self.entries) == self.capacity:
            old, _ = self.buckets[self.min_freq].popitem(last=False)
            del self.entries[old]
        # a new key always starts at frequency 1
        self.entries[key] = [value, 1]
        self.buckets[1][key] = None
        self.min_freq = 1


lfu = LFUCache(2)
lfu.put(1, 1)
lfu.put(2, 2)
assert lfu.get(1) == 1
lfu.put(3, 3)
assert lfu.get(2) == -1 and lfu.get(3) == 3
lfu.put(4, 4)
assert [lfu.get(1), lfu.get(3), lfu.get(4)] == [-1, 3, 4]
print("ok")

ok


### 3.4.3 [1472. Design Browser History](https://leetcode.com/problems/design-browser-history/) — ★ · Medium

**Problem.** `visit(url)` clears forward history; `back(steps)` and `forward(steps)` move and return the current URL.  
**Signal.** A list plus a cursor and a "last valid" index (a doubly linked list also works).  
*New here: Lazy deletion: invalidating a tail by moving a boundary instead of deleting.*

**Fine print.** Ask before coding:
- If `back` asks for more steps than exist? → Go as far as possible.
- Does `visit` delete the forward history? → Yes.

> 💡 **Key insight.** Forward history is everything after the cursor up to `last`; visiting a page just moves `last` back to the cursor, discarding the rest without touching it.

**Brute force →** Truncate the list on every visit → O(n) per visit. A `last` boundary makes every operation O(1).

**Steps**
1. Keep `history`, cursor `cur`, and `last` (the furthest valid page).
2. `visit`: write at `cur + 1` (append if needed), set `last = cur`.
3. `back`: `cur = max(0, cur − steps)`.
4. `forward`: `cur = min(last, cur + steps)`.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(number of visits).

In [81]:
class BrowserHistory:
    """Back/forward browser history."""

    def __init__(self, homepage: str) -> None:
        self.history = [homepage]
        # cur: current page; last: furthest page still reachable by forward
        self.cur = self.last = 0

    def visit(self, url: str) -> None:
        self.cur += 1
        # overwrite stale forward pages instead of deleting them
        if self.cur == len(self.history):
            self.history.append(url)
        else:
            self.history[self.cur] = url
        self.last = self.cur

    def back(self, steps: int) -> str:
        self.cur = max(0, self.cur - steps)
        return self.history[self.cur]

    def forward(self, steps: int) -> str:
        self.cur = min(self.last, self.cur + steps)
        return self.history[self.cur]


b = BrowserHistory("leetcode.com")
for url in ["google.com", "facebook.com", "youtube.com"]:
    b.visit(url)
assert b.back(1) == "facebook.com" and b.back(1) == "google.com"
assert b.forward(1) == "facebook.com"
b.visit("linkedin.com")
assert b.forward(2) == "linkedin.com"
assert b.back(2) == "google.com" and b.back(7) == "leetcode.com"
print("ok")

ok


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Matching stack | brackets, paths, nested decode, expression | push opens/context, pop on close | O(n) |
| Monotonic stack | next greater/smaller, span, histogram | pop while new element breaks the order | O(n) |
| Contribution counting | sum over all subarrays of min/max | previous/next smaller distances | O(n) |
| Lower-bound binary search | sorted lookup, insert position, first/last | `while lo < hi`, `hi = mid` on true | O(log n) |
| Rotated search | rotated sorted array | one half is always sorted | O(log n) |
| Search on answer | "minimum capacity / speed / day such that…" | binary search the value, greedy feasibility | O(n log R) |
| Partition search | median of two arrays | binary search the cut of the shorter array | O(log min) |
| Dummy head | head may change | `dummy = ListNode(0, head)` | — |
| Slow/fast | middle, cycle, k-th from end, duplicate number | 2× speed or fixed gap | O(n), O(1) |
| Reversal | reverse all/part/groups, palindrome, reorder | save next → rewire → advance | O(n), O(1) |
| Map + doubly linked list | LRU / LFU caches | O(1) lookup + O(1) reorder | O(1) |